# 🎸 Stems para remezclar (Colab · GPU T4)

Separa cada tema en stems **sumables** (la suma de todos = la mezcla, por resta exacta), los afina a tu referencia **antes** de separar y deja todo listo para Logic o Ableton.

**Cómo se usa**
1. Sube tus audios (FLAC, WAV o MP3) a `Drive/stems/entrada`.
2. Activa la GPU: *Entorno de ejecución › Cambiar tipo de entorno de ejecución › T4 GPU*.
3. Ajusta **solo** la celda de configuración de abajo.
4. *Entorno de ejecución › Ejecutar todo*.
5. Resultados en `Drive/stems/salida/<tema>/`.

Los temas ya terminados se saltan solos. Para rehacer uno, ponlo en **REPROCESAR** (configuración): su salida anterior se mueve a `salida/_anteriores/`, no se borra.

Afinación a mano por tema (opcional): crea `entrada/<tema>.afinacion.txt` con una línea `corregir -8.0`, `detectado 442.1` o `no_afinar`.

**Clic:** se genera al final de cada tema en `salida/<tema>/clic/`. Para temas ya procesados usa `ACCION = "solo clic"` (no separa ni afina, no necesita GPU). Marcas corregidas en Reaper: `entrada/<tema>.clic.csv` (detalles en `clic/clic_informe.txt`).

## ⚙️ Configuración

In [ ]:
#@title ⚙️ Configuración (lo único que se ajusta) { display-mode: "form" }
#@markdown ### 0 · Qué hacer
#@markdown **procesar todo** = afinar, separar stems y generar el clic de cada tema nuevo (GPU)
#@markdown <br>**solo clic** = generar el clic de temas YA procesados, sin separar ni afinar de nuevo (no necesita GPU)
ACCION = "procesar todo"  #@param ["procesar todo", "solo clic"]

#@markdown ### 1 · Modo
#@markdown **karaoke** = solo voz principal / coros / instrumental (rápido)
#@markdown <br>**completo** = todos los stems: voces, batería, bajo, guitarras, piano, otros (lento)
MODO = "completo"  #@param ["karaoke", "completo"]

#@markdown ### 2 · Guitarras (solo en modo completo; en karaoke se ignoran y no se descargan sus modelos)
#@markdown Modelo que separa las guitarras del resto (compara de oído y fija el ganador):
MODELO_GUITARRA = "becruily"  #@param ["becruily", "chencfd"]
#@markdown Orden: "punteo primero" separa el solo antes que acústica/eléctrica (recomendado para solos clean):
ORDEN_GUITARRAS = "punteo primero"  #@param ["punteo primero", "timbre primero"]
#@markdown Separar guitarras acústicas de eléctricas (descarga una vez un modelo de 1,37 GB):
SEPARAR_ACUSTICA_ELECTRICA = True  #@param {type:"boolean"}

#@markdown ### 3 · Afinación (se mide y se corrige ANTES de separar)
#@markdown Activar la corrección de afinación:
AFINAR = True  #@param {type:"boolean"}
#@markdown Frecuencia de referencia destino (La4):
REFERENCIA = "440 Hz"  #@param ["432 Hz", "440 Hz", "442 Hz", "444 Hz", "otra (usar el deslizador)"]
#@markdown Solo si elegiste "otra":
OTRA_REFERENCIA_HZ = 440  #@param {type:"slider", min:400, max:480, step:0.5}
#@markdown Solo corrige si la desviación supera este umbral (cents):
UMBRAL_CENTS = 1.5  #@param [0.5, 1.0, 1.5, 2.0, 3.0] {type:"raw"}

#@markdown ### 4 · Formato de salida
#@markdown FLAC 24-bit para el DAW (la suma de stems = la mezcla); MP3 solo para escuchar (rompe la suma exacta):
FORMATO = "FLAC 24-bit"  #@param ["FLAC 24-bit", "MP3 320"]
#@markdown Frecuencia de muestreo de los archivos (48 kHz = remuestreo de alta calidad al final):
FRECUENCIA = "44.1 kHz"  #@param ["44.1 kHz", "48 kHz"]

#@markdown ### 5 · Reprocesar temas ya terminados
#@markdown Nombres de temas separados por coma (como su carpeta en salida/), o «todos». Vacío = no rehacer nada.
#@markdown La salida anterior se MUEVE a salida/_anteriores/<tema>_<fecha>/ (no se borra). Vacíalo después.
REPROCESAR = ""  #@param {type:"string"}

#@markdown ### 6 · Kaggle (solo si corres en Kaggle)
#@markdown Nombre del Dataset con tus temas, tal como aparece en /kaggle/input/. Vacío = si hay uno solo, se detecta.
KAGGLE_DATASET = ""  #@param {type:"string"}

#@markdown ### 7 · Carpetas en Drive (solo Colab; normalmente no hay que tocarlas)
CARPETA_ENTRADA = "/content/drive/MyDrive/stems/entrada"  #@param {type:"string"}
CARPETA_SALIDA = "/content/drive/MyDrive/stems/salida"  #@param {type:"string"}
CARPETA_MODELOS = "/content/drive/MyDrive/stems/modelos"  #@param {type:"string"}

# --- Valores derivados (no editar) ---
if REFERENCIA.startswith("otra"):
    DESTINO_HZ = float(OTRA_REFERENCIA_HZ)
else:
    DESTINO_HZ = float(REFERENCIA.split()[0])
if not AFINAR:
    DESTINO_HZ = None
UMBRAL_CENTS = float(UMBRAL_CENTS)
_REPROCESAR = {t.strip().casefold() for t in REPROCESAR.replace(";", ",").split(",") if t.strip()}


def en_reprocesar(tema):
    """¿Está este tema en REPROCESAR? (sin distinguir mayúsculas; acepta el nombre con extensión)."""
    if "todos" in _REPROCESAR:
        return True
    tema = tema.casefold()
    return any(t == tema or t.rsplit(".", 1)[0] == tema for t in _REPROCESAR)


# minutos de proceso por minuto de canción en una T4 (aproximado; el resumen final muestra el real)
MIN_POR_MIN_AUDIO = 1.0 if MODO == "karaoke" else (2.5 if SEPARAR_ACUSTICA_ELECTRICA else 2.0)


def resumen_configuracion():
    L = ["Configuración elegida:"]
    if ACCION == "solo clic":
        L.append("  · SOLO CLIC: se genera el clic de los temas ya procesados (sin separar ni afinar; corre sin GPU)")
        L.append(f"  · Salida del clic: {FORMATO} a {FRECUENCIA} (igual que los stems)")
        return "\n".join(L)
    if MODO == "karaoke":
        L.append("  · Modo karaoke: 01 Voz principal, 02 Coros, 03 Instrumental (rápido)")
        L.append("    (los parámetros de guitarras se ignoran y no se descargan sus modelos)")
    else:
        L.append("  · Modo completo: voces, batería, bajo, guitarras, piano y otros (lento)")
        L.append(f"  · Guitarras: modelo {MODELO_GUITARRA} · orden «{ORDEN_GUITARRAS}» · "
                 f"acústica/eléctrica: {'sí' if SEPARAR_ACUSTICA_ELECTRICA else 'no'}")
    if DESTINO_HZ is None:
        L.append("  · Afinación: no se mide ni se corrige")
    else:
        L.append(f"  · Afinación: a {DESTINO_HZ:g} Hz, solo si la desviación supera ±{UMBRAL_CENTS:g} ¢")
    L.append(f"  · Salida: {FORMATO} a {FRECUENCIA}" + (" (⚠ MP3 rompe la suma exacta)" if FORMATO.startswith("MP3") else ""))
    L.append("  · Clic: se genera al final de cada tema (Beat This!), en <tema>/clic/")
    if _REPROCESAR:
        L.append(f"  · Reprocesar: {', '.join(sorted(_REPROCESAR))} (la salida anterior se mueve a salida/_anteriores/)")
    L.append(f"  · Tiempo estimado: ~{MIN_POR_MIN_AUDIO:g} min por cada minuto de canción "
             f"(un tema de 4 min ≈ {4 * MIN_POR_MIN_AUDIO:.0f} min), más la descarga de modelos la primera vez")
    return "\n".join(L)


print(resumen_configuracion())


## 1 · Preparar
Detecta si corre en Colab o en Kaggle, revisa la GPU, prepara las carpetas (Drive en Colab; Dataset y `/kaggle/working` en Kaggle) y revisa el espacio.

In [ ]:
#@title 1 · Preparar: entorno (Colab o Kaggle), GPU, carpetas y espacio
import os, sys, json, math, time, shutil, subprocess, traceback
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Kaggle trae dos T4: se usa solo la primera (cuda:0)


class Detenido(Exception):
    """Error con mensaje claro para ti: detiene "Ejecutar todo" en esta celda."""


EXTENSIONES = {".flac", ".wav", ".mp3"}
LIMITE_KAGGLE_MB = 19_500     # /kaggle/working admite ~20 GB; se deja un margen
MB_MODELOS = {"base": 1_700, "guitarras": 1_800, "acustica": 1_400}  # aproximados (voz+karaoke+demucs / 2 guitarra / mega)


def detectar_entorno():
    """'kaggle' | 'colab' | 'local'."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE") or Path("/kaggle/input").is_dir():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or Path("/content").is_dir():
        return "colab"
    return "local"


def rutas_kaggle(dataset="", raiz=Path("/kaggle"), solo_clic=False):
    """Entrada = el Dataset en /kaggle/input (si hay uno solo se detecta).
    En /kaggle/working (el Output) quedan SOLO stems/zips/<tema>.zip, stems/salida/progreso.txt y errores.txt.
    Stems sueltos, modelos y temporales van a /kaggle/temp (o /tmp): no ocupan el Output."""
    entrada_base = raiz / "input"
    if solo_clic:  # «solo clic»: se buscan zips ya procesados en TODOS los Inputs (Datasets u Output anterior)
        if not entrada_base.is_dir() or not any(entrada_base.iterdir()):
            raise Detenido("«solo clic» en Kaggle: agrega como Input el Dataset con los zips de tus temas "
                           "(o el Output de una versión anterior de este notebook) con «+ Add Input».")
        entrada = entrada_base
    elif dataset.strip():
        candidatos = [entrada_base / dataset.strip()] + sorted(entrada_base.glob(f"**/{dataset.strip()}"))
        entrada = next((c for c in candidatos if c.is_dir()), None)
        if entrada is None:
            disponibles = sorted(p.name for p in entrada_base.iterdir() if p.is_dir()) if entrada_base.is_dir() else []
            raise Detenido(f"No encuentro el Dataset «{dataset}» en {entrada_base}. Disponibles: {disponibles}.\n"
                           "Agrégalo con «+ Add Input» (panel derecho) o corrige KAGGLE_DATASET.")
    else:
        datasets = sorted(p for p in entrada_base.iterdir() if p.is_dir()) if entrada_base.is_dir() else []
        if len(datasets) != 1:
            raise Detenido(f"Hay {len(datasets)} Datasets en {entrada_base} ({[p.name for p in datasets]}). "
                           "Agrega el de tus temas con «+ Add Input», o escribe su nombre en KAGGLE_DATASET.")
        entrada = datasets[0]
    trabajo_base = raiz / "temp"
    try:
        trabajo_base.mkdir(parents=True, exist_ok=True)
    except OSError:
        trabajo_base = Path("/tmp")
    return {"ENTRADA": entrada, "SALIDA": trabajo_base / "stems/salida", "MODELOS": trabajo_base / "stems/modelos",
            "TRABAJO": trabajo_base / "stems_trabajo", "TEMP": trabajo_base, "WORKING": raiz / "working",
            "SALIDA_INFO": raiz / "working/stems/salida", "ZIPS": raiz / "working/stems/zips"}


MB_CLIC_TEMA = 15  # zip de la carpeta clic/ (clic.flac es casi todo silencio) + marcas + MIDI


def temas_procesados_colab(salida):
    """«solo clic» en Colab: carpetas completas (con manifiesto) en salida/."""
    return sorted((p for p in salida.iterdir() if p.is_dir() and not p.name.startswith("_")
                   and (p / ".proceso.json").exists()), key=lambda p: p.name.lower())


def zips_procesados_kaggle(entrada):
    """«solo clic» en Kaggle: zips <tema>.zip con <tema>/.proceso.json, en cualquier Input (recursivo)."""
    import zipfile
    hallados = {}
    for z in sorted(entrada.rglob("*.zip")):
        if "_anteriores" in z.parts or z.name.endswith(".clic.zip"):
            continue
        try:
            with zipfile.ZipFile(z) as zf:
                if f"{z.stem}/.proceso.json" not in zf.namelist():
                    continue
        except (zipfile.BadZipFile, OSError):
            continue
        if z.stem in hallados:
            print(f"⚠ {z.stem}: hay más de un zip; uso el más reciente")
            if z.stat().st_mtime <= hallados[z.stem].stat().st_mtime:
                continue
        hallados[z.stem] = z
    return [hallados[k] for k in sorted(hallados, key=str.lower)]


def revisar_espacio_solo_clic_kaggle(working, temp, zips):
    """«solo clic»: en working, un <tema>.clic.zip pequeño por tema; en temp, el zip más grande descomprimido."""
    necesario_w = MB_CLIC_TEMA * len(zips)
    libre_w = min(LIMITE_KAGGLE_MB - tamano_arbol_mb(working), shutil.disk_usage(working).free / 1e6)
    necesario_t = 1.2 * max((z.stat().st_size / 1e6 for z in zips), default=0) + 300  # + Beat This! y margen
    libre_t = shutil.disk_usage(temp).free / 1e6
    print(f"✓ Solo clic · working: ~{necesario_w:.0f} MB de clics (disponible {libre_w / 1000:.1f} GB) · "
          f"temp: ~{necesario_t / 1000:.1f} GB para descomprimir de a un tema (libre {libre_t / 1000:.1f} GB)")
    if necesario_w > libre_w or necesario_t > libre_t:
        raise Detenido("No hay espacio suficiente para «solo clic» (ver arriba).")


def listar_temas(entrada, recursivo):
    patron = entrada.rglob("*") if recursivo else entrada.iterdir()
    return sorted((p for p in patron if p.is_file() and p.suffix.lower() in EXTENSIONES), key=lambda p: p.name.lower())


def tamano_arbol_mb(carpeta):
    if not carpeta.exists():
        return 0.0
    return sum(f.stat().st_size for f in carpeta.rglob("*") if f.is_file()) / 1e6


def mb_modelos_previstos(modelos_dir):
    """Lo que falta por descargar (en una sesión nueva de Kaggle, todo)."""
    ya = tamano_arbol_mb(modelos_dir)
    total = MB_MODELOS["base"] + (MB_MODELOS["guitarras"] if MODO == "completo" else 0) \
        + (MB_MODELOS["acustica"] if MODO == "completo" and SEPARAR_ACUSTICA_ELECTRICA else 0)
    return max(0.0, total - ya)


def revisar_espacio_kaggle(working, temp, mb_zips, mb_tema_max, mb_modelos):
    """Antes de empezar: /kaggle/working solo guarda los zips (tope ~20 GB); /kaggle/temp guarda los
    modelos y, de a un tema por vez, sus stems sueltos + intermedios (se borran tras verificar su zip)."""
    usado = tamano_arbol_mb(working)
    libre_w = min(LIMITE_KAGGLE_MB - usado, shutil.disk_usage(working).free / 1e6)
    print(f"✓ /kaggle/working (Output): zips ~{mb_zips / 1000:.1f} GB · usado {usado / 1000:.1f} GB · "
          f"disponible ~{libre_w / 1000:.1f} GB de {LIMITE_KAGGLE_MB / 1000:.1f}")
    if mb_zips > libre_w:
        raise Detenido("Los zips no caben en /kaggle/working (tope ~20 GB): sube menos temas por Dataset, "
                       "usa modo karaoke o formato MP3.")
    necesario_t = mb_modelos + 2 * mb_tema_max  # modelos + stems sueltos de un tema + intermedios
    libre_t = shutil.disk_usage(temp).free / 1e6
    print(f"✓ /kaggle/temp: necesario ~{necesario_t / 1000:.1f} GB (modelos {mb_modelos / 1000:.1f} + "
          f"un tema a la vez {2 * mb_tema_max / 1000:.1f}) · libre {libre_t / 1000:.1f} GB")
    if necesario_t > libre_t:
        raise Detenido(f"No hay espacio en {temp} para modelos + un tema (~{necesario_t / 1000:.1f} GB, libres "
                       f"{libre_t / 1000:.1f} GB): prueba con modo karaoke o sin separar acústica/eléctrica.")


def duracion_segundos(ruta):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", str(ruta)], capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return 300.0  # si no se puede leer, suponer 5 min


def estimar_mb_tema(segundos):
    """Tamaño aproximado de la carpeta de un tema (stems + mezcla afinada + clics)."""
    n_stems = (3 if MODO == "karaoke" else 9) + 1 + 1  # stems + mezcla afinada + clic
    if FORMATO.startswith("MP3"):
        bytes_por_s = 320_000 / 8
    else:
        sr = 48000 if FRECUENCIA.startswith("48") else 44100
        bytes_por_s = sr * 2 * 3 * 0.6  # 24 bit estéreo, FLAC comprime ~40 %
    return n_stems * segundos * bytes_por_s / 1e6


# ============================ ejecución ============================
SOLO_CLIC = ACCION == "solo clic"
ENTORNO = detectar_entorno()
KAGGLE_SEGUNDO_PLANO = ENTORNO == "kaggle" and os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") == "Batch"
print(f"✓ Entorno: {ENTORNO}" + (" (en segundo plano: Save & Run All)" if KAGGLE_SEGUNDO_PLANO else ""))

# --- GPU (se revisa con nvidia-smi, antes de instalar nada) ---
_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if (_gpu is None or _gpu.returncode != 0 or "GPU" not in _gpu.stdout) and SOLO_CLIC:
    print("✓ Sin GPU: en «solo clic» no hace falta (Beat This! corre en CPU)")
elif _gpu is None or _gpu.returncode != 0 or "GPU" not in _gpu.stdout:
    if ENTORNO == "kaggle":
        raise Detenido("No hay GPU activa. En Kaggle: panel derecho › Session options › Accelerator › "
                       "«GPU T4 x2». Después: Save Version › Save & Run All.")
    raise Detenido(
        "No hay GPU activa.\n"
        "Actívala así: menú Entorno de ejecución > Cambiar tipo de entorno de ejecución > "
        "Acelerador por hardware: T4 GPU > Guardar.\n"
        "Después: Entorno de ejecución > Ejecutar todo.")
else:
    print("✓", _gpu.stdout.strip().splitlines()[0], "(se usa solo esta: cuda:0)")

# --- Carpetas ---
if ENTORNO == "kaggle":
    _r = rutas_kaggle(KAGGLE_DATASET, solo_clic=SOLO_CLIC)
    ENTRADA, SALIDA, MODELOS, TRABAJO, WORKING = (_r[k] for k in ("ENTRADA", "SALIDA", "MODELOS", "TRABAJO", "WORKING"))
    TEMP, SALIDA_INFO, ZIPS = _r["TEMP"], _r["SALIDA_INFO"], _r["ZIPS"]
    try:  # los modelos se descargan de Internet en cada ejecución
        import urllib.request
        urllib.request.urlopen("https://huggingface.co", timeout=15)
    except Exception:
        raise Detenido("Kaggle no tiene Internet en esta sesión. Panel derecho › Session options › "
                       "Internet: On (requiere teléfono verificado en tu cuenta).")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ENTRADA, SALIDA, MODELOS = Path(CARPETA_ENTRADA), Path(CARPETA_SALIDA), Path(CARPETA_MODELOS)
    TRABAJO = Path("/content/trabajo")  # temporales locales, fuera de Drive
    WORKING = TEMP = ZIPS = None
    SALIDA_INFO = SALIDA  # en Colab progreso.txt y errores.txt van junto a los temas
    ENTRADA.mkdir(parents=True, exist_ok=True)
for _c in (SALIDA, MODELOS, TRABAJO, SALIDA_INFO) + ((ZIPS,) if ZIPS else ()):
    _c.mkdir(parents=True, exist_ok=True)

# --- Tiempo y espacio ---
def tema_terminado(tema):
    """Colab: su carpeta tiene manifiesto. Kaggle: ya existe su zip en el Output."""
    if ZIPS is not None:
        return (ZIPS / f"{tema}.zip").exists()
    return (SALIDA / tema / ".proceso.json").exists()


if SOLO_CLIC:
    TEMAS = zips_procesados_kaggle(ENTRADA) if ENTORNO == "kaggle" else temas_procesados_colab(SALIDA)
    print(f"✓ Solo clic: {len(TEMAS)} tema(s) ya procesado(s) en {ENTRADA if ENTORNO == 'kaggle' else SALIDA}")
    for p in TEMAS:
        print("   ·", p.name)
    if not TEMAS:
        raise Detenido("No encontré temas ya procesados. " + (
            "En Kaggle agrega como Input el Dataset con los <tema>.zip (o el Output de una versión anterior)."
            if ENTORNO == "kaggle" else f"En Colab deben estar en {SALIDA}/<tema>/ con su .proceso.json."))
    if ENTORNO == "kaggle":
        revisar_espacio_solo_clic_kaggle(WORKING, TEMP, TEMAS)
else:
    TEMAS = listar_temas(ENTRADA, recursivo=ENTORNO == "kaggle")
    print(f"✓ Entrada: {ENTRADA} ({len(TEMAS)} archivos de audio)")
    for p in TEMAS:
        print("   ·", p.relative_to(ENTRADA))
    if not TEMAS:
        raise Detenido(f"No hay audios (FLAC, WAV o MP3) en {ENTRADA}. Súbelos ahí y vuelve a ejecutar.")
    _repetidos = sorted({p.stem for p in TEMAS if [q.stem for q in TEMAS].count(p.stem) > 1})
    if _repetidos:
        raise Detenido(f"Hay temas con el mismo nombre en distintas carpetas: {_repetidos}. Renómbralos.")
    print(f"✓ Salida: {SALIDA} · modelos: {MODELOS} · temporales: {TRABAJO}"
          + (f"\n✓ Output de Kaggle: {ZIPS} (zips) y {SALIDA_INFO} (progreso.txt, errores.txt)" if ZIPS else ""))

    _pendientes = [p for p in TEMAS if not tema_terminado(p.stem) or en_reprocesar(p.stem)]
    _duraciones = {p: duracion_segundos(p) for p in _pendientes}
    _estimado = sum(estimar_mb_tema(d) for d in _duraciones.values())
    _minutos_audio = sum(_duraciones.values()) / 60
    print(f"✓ Tiempo estimado para los pendientes: ~{_minutos_audio * MIN_POR_MIN_AUDIO:.0f} min "
          f"({_minutos_audio:.1f} min de audio × {MIN_POR_MIN_AUDIO:g}; aproximado)")
    if ENTORNO == "kaggle":
        revisar_espacio_kaggle(WORKING, TEMP, _estimado, max(map(estimar_mb_tema, _duraciones.values()), default=0),
                               mb_modelos_previstos(MODELOS))
    else:
        if (MODO == "completo" and SEPARAR_ACUSTICA_ELECTRICA
                and not (MODELOS / "bs_roformer_mega_53stem_mvsep.ckpt").exists()):
            _estimado += 1400  # modelo de guitarra acústica (53 stems, 1,37 GB), se descarga una vez
        _libre = shutil.disk_usage(SALIDA).free / 1e6
        print(f"✓ Espacio libre en Drive: {_libre / 1000:.1f} GB · "
              f"estimado para {len(_pendientes)} tema(s) pendiente(s): {_estimado / 1000:.2f} GB "
              f"(~{_estimado / max(1, len(_pendientes)):.0f} MB por tema)")
        if _libre < 2000:
            print("⚠ ATENCIÓN: quedan menos de 2 GB libres en Drive.")
        if _estimado > _libre:
            raise Detenido("No alcanza el espacio en Drive para los temas pendientes. "
                           "Libera espacio o deja menos archivos en la carpeta de entrada.")


## 2 · Instalar
audio-separator **fijo en 0.44.5** (el arreglo de la celda 6 depende de esa versión), Rubber Band 3 (pylibrb), soxr.

In [ ]:
#@title 2 · Instalar programas (2–4 min la primera vez)
# ⚠ audio-separator queda FIJO en 0.44.5. El arreglo de la celda 6 (monkeypatch de
#   MelBandRoformer) depende de esa versión exacta: si lo actualizas, puede romperse.
_paquetes = ["audio-separator[gpu]==0.44.5", "soxr", "huggingface_hub",
             "audioread"]  # audioread: audio-separator lo importa y librosa 1.x ya no lo trae
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_paquetes], capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stdout[-3000:], _r.stderr[-3000:])
    raise Detenido("Falló la instalación de paquetes (ver arriba). Reinicia el entorno y vuelve a ejecutar todo.")
# pylibrb (Rubber Band 3) va aparte: si no se instala, la celda 5 usa el plan B (compilar rubberband)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylibrb==0.1.2"], capture_output=True, text=True)
if _r.returncode != 0:
    print("⚠ pylibrb no se pudo instalar; se usará el plan B (compilar Rubber Band):", _r.stderr[-500:])
# Clic: Beat This! (CPJKU, fijado a un commit) y mido para el MIDI. torchaudio ya viene en Colab y Kaggle.
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mido",
                     "beat_this @ git+https://github.com/CPJKU/beat_this@b95c8ab"], capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stdout[-2000:], _r.stderr[-2000:])
    raise Detenido("No se pudo instalar Beat This! (clic). Revisa que haya Internet y vuelve a ejecutar todo.")
try:
    import torchaudio  # noqa: F401  (Beat This! lo usa para el espectrograma)
except Exception as _e:
    raise Detenido(f"torchaudio no está disponible ({_e}); Beat This! lo necesita para el clic.")
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

from importlib.metadata import version as _version
try:
    _v_rb = _version("pylibrb")
except Exception:
    _v_rb = "no instalado (plan B)"
print("✓ audio-separator", _version("audio-separator"), "· pylibrb", _v_rb, "· Python", sys.version.split()[0])


## 3 · Utilidades
Borrado seguro (solo dentro de las carpetas de trabajo, nunca con comodines, jamás en `entrada/`), lectura con **una sola decodificación** y escritura FLAC 24-bit / MP3.

In [ ]:
#@title 3 · Utilidades: borrado seguro, lectura y escritura de audio
import numpy as np
import soundfile as sf
import soxr

SR = 44100  # la separación y la afinación trabajan siempre a 44,1 kHz
SR_SALIDA = 48000 if FRECUENCIA.startswith("48") else 44100
ES_MP3 = FORMATO.startswith("MP3")
EXT = ".mp3" if ES_MP3 else ".flac"


# ---------- Borrado seguro: la ÚNICA forma de borrar en este notebook ----------
def _dentro(ruta, base):
    try:
        ruta.relative_to(base)
        return True
    except ValueError:
        return False


def borrar_seguro(ruta, tema_en_curso=None):
    """Borra UN archivo o UNA carpeta con nombre exacto (nunca comodines).
    Solo se permite dentro de la carpeta temporal de trabajo o de salida/<tema en curso>/.
    En Kaggle, además, la carpeta COMPLETA del tema en curso, porque vive en /kaggle/temp y
    ya quedó verificada en su zip. Jamás toca la entrada, los modelos, /kaggle/input ni /kaggle/working."""
    texto = str(ruta)
    if any(c in texto for c in "*?[]"):
        raise PermissionError(f"borrar_seguro: no se aceptan comodines: {texto}")
    r = Path(ruta).resolve()
    prohibidas = [ENTRADA.resolve(), MODELOS.resolve(), Path("/kaggle/input").resolve()]
    if globals().get("WORKING") is not None:
        prohibidas.append(WORKING.resolve())
    for prohibida in prohibidas:
        if _dentro(r, prohibida) or _dentro(prohibida, r):
            raise PermissionError(f"borrar_seguro: {r} está en (o contiene) una carpeta protegida: {prohibida}")
    if any(r != p and _dentro(r, p) for p in [TRABAJO.resolve()]):
        pass
    elif tema_en_curso and _dentro(r, (SALIDA / tema_en_curso).resolve()):
        carpeta_tema = (SALIDA / tema_en_curso).resolve()
        en_temp = globals().get("TEMP") is not None and _dentro(carpeta_tema, TEMP.resolve())
        if r == carpeta_tema and not en_temp:
            raise PermissionError(f"borrar_seguro: la carpeta de salida completa solo se borra en /kaggle/temp: {r}")
    else:
        raise PermissionError(f"borrar_seguro: {r} está fuera de las carpetas de trabajo permitidas")
    if not r.exists() and not r.is_symlink():
        return
    if r.is_dir() and not r.is_symlink():
        shutil.rmtree(r)
    else:
        r.unlink()


# ---------- Avance visible (barras, línea fija y progreso.txt en Drive) ----------
def _hms(segundos):
    segundos = int(max(0, segundos))
    return f"{segundos // 3600}:{segundos % 3600 // 60:02d}:{segundos % 60:02d}" if segundos >= 3600 \
        else f"{segundos // 60}:{segundos % 60:02d}"


class Progreso:
    """Tres barras que se actualizan en el mismo lugar (tqdm: temas, etapas del tema, segmentos),
    una línea fija de texto (por si el widget no se ve en el celular) y Drive/stems/salida/progreso.txt,
    que se reescribe en cada etapa para revisar el avance desde otro dispositivo."""
    INTERVALO_S = 1.0         # refresco mínimo de la línea fija
    INTERVALO_ARCHIVO_S = 20  # dentro de una etapa, reescribir progreso.txt cada tanto

    def __init__(self, n_temas, archivo=None, mostrar=True, lineas_texto=False):
        # lineas_texto: además, una línea de log por etapa (Kaggle en segundo plano no muestra widgets)
        self.n_temas, self.archivo, self.mostrar, self.lineas_texto = n_temas, archivo, mostrar, lineas_texto
        self.i_tema, self.tema, self.etapas_total, self.i_etapa, self.etapa = 0, "", 1, 0, ""
        self.seg, self.seg_total = 0, 0
        self.t0 = self.t_tema = self.t_etapa = time.time()
        self._t_linea = self._t_archivo = 0.0
        self.barras = self.linea = None
        if mostrar:
            try:
                from tqdm.auto import tqdm
                fmt = "{desc} {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
                self.barras = [tqdm(total=n_temas, desc="Temas", bar_format=fmt),
                               tqdm(total=1, desc="Etapas", bar_format=fmt),
                               tqdm(total=1, desc="Segmentos", bar_format=fmt)]
            except Exception:
                self.barras = None
            try:
                from IPython import get_ipython
                from IPython.display import display
                if get_ipython() is not None:  # solo dentro de Colab/Jupyter
                    self.linea = display({"text/plain": "▶ preparando…"}, raw=True, display_id=True)
            except Exception:
                self.linea = None

    # --- eventos ---
    def nuevo_tema(self, i, nombre, etapas_total):
        self.i_tema, self.tema, self.etapas_total, self.i_etapa, self.etapa = i, nombre, max(1, etapas_total), 0, ""
        self.seg = self.seg_total = 0
        self.t_tema = time.time()
        if self.barras:
            self.barras[0].n = i - 1; self.barras[0].refresh()
            self.barras[1].reset(total=self.etapas_total); self.barras[1].set_description(f"Etapas · {nombre[:24]}")
            self.barras[2].reset(total=1)
        if self.lineas_texto:
            print(f"[{time.strftime('%H:%M:%S')}] ━━ Tema {i}/{self.n_temas}: {nombre}", flush=True)
        self._actualizar(forzar=True)

    def nueva_etapa(self, nombre):
        self.i_etapa += 1
        if self.i_etapa > self.etapas_total:  # etapas extra (p. ej. lazo cerrado de afinación)
            self.etapas_total = self.i_etapa
            if self.barras:
                self.barras[1].total = self.etapas_total
        self.etapa, self.seg, self.seg_total = nombre, 0, 0
        self.t_etapa = time.time()
        if self.barras:
            self.barras[1].n = self.i_etapa - 1; self.barras[1].refresh()
            self.barras[2].reset(total=1); self.barras[2].set_description(f"Segmentos · {nombre[:24]}")
        if self.lineas_texto:
            print(f"[{time.strftime('%H:%M:%S')}] {self.texto()}", flush=True)
        self._actualizar(forzar=True)

    def segmento(self, k, total):
        self.seg, self.seg_total = k, total
        if self.barras:
            if self.barras[2].total != total:
                self.barras[2].reset(total=total)
            self.barras[2].n = k; self.barras[2].refresh()
        self._actualizar()

    def fin_tema(self):
        if self.barras:
            self.barras[1].n = self.etapas_total; self.barras[1].refresh()
            self.barras[0].n = self.i_tema; self.barras[0].refresh()
        self.i_etapa = self.etapas_total
        self._actualizar(forzar=True)

    def cerrar(self, resumen):
        if self.barras:
            for b in self.barras:
                b.close()
        if self.linea:
            self.linea.update({"text/plain": "✓ Terminado. Resumen abajo y en salida/progreso.txt"}, raw=True)
        self._escribir("PROCESO TERMINADO · " + time.strftime("%Y-%m-%d %H:%M:%S") + "\n\n" + resumen)

    # --- salida ---
    def porcentaje_tema(self):
        dentro = (self.seg / self.seg_total) if self.seg_total else 0.0
        return 100.0 * min(1.0, (max(self.i_etapa - 1, 0) + dentro) / self.etapas_total)

    def texto(self):
        ahora = time.time()
        pct = self.porcentaje_tema()
        trans = ahora - self.t_tema
        resta = trans * (100 - pct) / pct if pct > 1 else None
        seg = f" · segmento {self.seg}/{self.seg_total}" if self.seg_total else ""
        return (f"▶ Tema {self.i_tema}/{self.n_temas} «{self.tema}» · etapa {self.i_etapa}/{self.etapas_total}: "
                f"{self.etapa}{seg} · {pct:.0f} % · {_hms(trans)} transcurrido"
                + (f" · ~{_hms(resta)} restante" if resta is not None else "")
                + f" · total {_hms(ahora - self.t0)}")

    def _actualizar(self, forzar=False):
        ahora = time.time()
        if self.linea and (forzar or ahora - self._t_linea >= self.INTERVALO_S):
            self.linea.update({"text/plain": self.texto()}, raw=True)
            self._t_linea = ahora
        if forzar or ahora - self._t_archivo >= self.INTERVALO_ARCHIVO_S:
            self._escribir(f"Actualizado: {time.strftime('%Y-%m-%d %H:%M:%S')}\n{self.texto()}\n")
            self._t_archivo = ahora

    def _escribir(self, contenido):
        if not self.archivo:
            return
        try:  # se escribe en un .parcial y se renombra: nunca queda a medio escribir
            parcial = Path(str(self.archivo) + ".parcial")
            parcial.write_text(contenido, encoding="utf-8")
            os.replace(parcial, self.archivo)
        except OSError as e:
            print("  (no pude escribir progreso.txt:", e, ")")


PROGRESO = None  # lo crea la celda 9


def avisar_segmento(k, total):
    if PROGRESO is not None:
        PROGRESO.segmento(k, total)


# ---------- Tiempos por etapa ----------
class Etapa:
    """Mide el tiempo de una etapa, la anuncia al avance y deja UNA línea al terminar."""
    def __init__(self, nombre, tiempos=None):
        self.nombre, self.tiempos = nombre, tiempos

    def __enter__(self):
        self.t0 = time.time()
        if PROGRESO is not None:
            PROGRESO.nueva_etapa(self.nombre)
        else:
            print(f"  ▸ {self.nombre}…", flush=True)
        return self

    def __exit__(self, *exc):
        dt = time.time() - self.t0
        if self.tiempos is not None:
            self.tiempos[self.nombre] = round(dt, 1)
        if exc[0] is None:
            print(f"    ✓ {self.nombre} ({_hms(dt)})", flush=True)
        return False


# ---------- Audio ----------
def dbfs(valor):
    return 20 * math.log10(max(float(valor), 1e-12))


def pico(x):
    return float(np.max(np.abs(x))) if x.size else 0.0


def leer_audio(ruta):
    """UNA sola decodificación (ffmpeg → float32 a su frecuencia nativa → soxr VHQ a 44,1 kHz).
    Devuelve un array (2, N) float32. Mono se duplica; más de 2 canales se mezcla a estéreo."""
    pr = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries",
                         "stream=sample_rate", "-of", "default=nw=1:nk=1", str(ruta)],
                        capture_output=True, text=True)
    sr_nativa = int(pr.stdout.strip().splitlines()[0])
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-map", "0:a:0", "-vn",
                        "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "2", "-"],
                       capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg no pudo leer {ruta.name}: {r.stderr.decode()[-500:]}")
    x = np.frombuffer(r.stdout, dtype=np.float32).reshape(-1, 2).T
    if sr_nativa != SR:
        x = soxr.resample(x.T, sr_nativa, SR, quality="VHQ").T
    return np.ascontiguousarray(x, dtype=np.float32), sr_nativa


def a_frecuencia_salida(x):
    """Remuestreo final de alta calidad (soxr VHQ, lineal: conserva la suma de stems)."""
    if SR_SALIDA == SR:
        return x
    return np.ascontiguousarray(soxr.resample(x.T, SR, SR_SALIDA, quality="VHQ").T, dtype=np.float32)


def escribir_audio(ruta_sin_ext, x, sr, mp3=None):
    """Escribe (2, N) float32. FLAC 24-bit (cuantiza solo aquí) o MP3 320 kbps (mp3=None: según FORMATO)."""
    mp3 = ES_MP3 if mp3 is None else mp3
    ruta = Path(str(ruta_sin_ext) + (".mp3" if mp3 else ".flac"))
    if mp3:
        r = subprocess.run(["ffmpeg", "-v", "error", "-y", "-f", "f32le", "-ar", str(sr), "-ac", "2",
                            "-i", "-", "-c:a", "libmp3lame", "-b:a", "320k", str(ruta)],
                           input=np.ascontiguousarray(x.T, dtype=np.float32).tobytes(), capture_output=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg no pudo escribir {ruta.name}: {r.stderr.decode()[-500:]}")
    else:
        sf.write(str(ruta), x.T, sr, subtype="PCM_24", format="FLAC")
    return ruta


def leer_escrito(ruta):
    """Relee un archivo ya escrito (para la prueba de cancelación). Devuelve (2, N) float64."""
    if ruta.suffix == ".flac":
        y, _ = sf.read(str(ruta), dtype="float64", always_2d=True)
        return y.T
    r = subprocess.run(["ffmpeg", "-v", "error", "-i", str(ruta), "-f", "f64le", "-ac", "2", "-"],
                       capture_output=True, check=True)
    return np.frombuffer(r.stdout, dtype=np.float64).reshape(-1, 2).T


def mm_ss(segundos):
    return f"{int(segundos // 60)}:{segundos % 60:04.1f}"

print("✓ Utilidades listas")


## 4 · Medición de afinación
Port a Python del detector de [centrail](https://github.com/Trainmusiq/centrail) (`engine/detect.mjs`).

In [ ]:
#@title 4 · Medición de afinación (port de centrail · engine/detect.mjs)
# Mismo método y parámetros que centrail (docs/especificacion.md §3). No se inventa otro:
# picos espectrales → desviación en cents módulo 100 con matemática circular → histograma
# de 1 ¢/bin + media circular local ±15 ¢; incertidumbre = diferencia entre ventanas
# pares e impares; deriva en 10 segmentos; filtro de persistencia de picos (v1.2).
#
# CASO LÍMITE CONOCIDO (heredado de centrail, se deja igual a propósito): el histograma
# tiene bins de 1 ¢ y la media circular local usa el CENTRO de cada bin, no la desviación
# exacta de cada pico. Un tono perfectamente puro cuyos picos caen todos en el mismo bin
# se mide en el centro de ese bin: error de hasta ±0,5 ¢ (p. ej. una senoidal a +7,000 ¢
# cae en el borde del bin [7, 8) y se mide +7,5 ¢). En música real los picos se reparten
# entre bins vecinos y el error baja a ~0,1–0,2 ¢. Mejora pendiente: refinar con la
# desviación exacta de cada pico, primero en centrail y después portarla aquí.
FFT_N = 32768
MAX_FRAMES = 240              # ventanas repartidas por todo el tema
FMIN, FMAX = 55.0, 5000.0     # rango útil para material armónico
NBINS = 100                   # 1 bin = 1 cent
SEGMENTS = 10                 # curva de deriva
PERSISTENCE_TOL_CENTS = 15
PERSISTENCE_TOP_K = 5
_HANN = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(FFT_N) / (FFT_N - 1))


def _envolver(d):
    """Lleva cents a [-50, 50) (equivale a ((d % 100) + 150) % 100 - 50 de JS)."""
    return ((np.asarray(d) % 100) + 150) % 100 - 50


def _redondear_js(x):
    """Math.round de JS (las mitades suben)."""
    return np.floor(np.asarray(x) + 0.5).astype(int)


def _picos_ventana(x, inicio, sr):
    """Picos de una ventana: (desviación ¢, peso, frecuencia Hz), del más fuerte al más débil."""
    mono = x[:, inicio:inicio + FFT_N].mean(axis=0) * _HANN
    espectro = np.fft.rfft(mono)
    mitad = FFT_N // 2
    mag = np.abs(espectro[:mitad])
    mag[0] = 0.0
    max_mag = mag[1:].max()
    if max_mag <= 0:
        return np.empty(0), np.empty(0), np.empty(0)
    umbral = max_mag * 0.002
    bin_hz = sr / FFT_N
    i_min = max(2, math.ceil(FMIN / bin_hz))
    i_max = min(mitad - 2, math.floor(FMAX / bin_hz))
    i = np.arange(i_min, i_max + 1)
    m = mag[i]
    es_pico = (m >= umbral) & (m > mag[i - 1]) & (m >= mag[i + 1])
    i, m = i[es_pico], m[es_pico]
    if i.size == 0:
        return np.empty(0), np.empty(0), np.empty(0)
    a, b, c = np.log(mag[i - 1] + 1e-12), np.log(m + 1e-12), np.log(mag[i + 1] + 1e-12)
    den = a - 2 * b + c
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(den != 0, 0.5 * (a - c) / den, 0.0)
    p = np.clip(p, -0.5, 0.5)
    f = (i + p) * bin_hz
    dev = _envolver(1200 * np.log2(f / 440.0))
    w = np.sqrt(m / max_mag)
    orden = np.argsort(-w, kind="stable")
    return dev[orden], w[orden], f[orden]


def _coinciden(fa, fb):
    a, b = fa[:PERSISTENCE_TOP_K], fb[:PERSISTENCE_TOP_K]
    if a.size == 0 or b.size == 0:
        return False
    return bool(np.any(np.abs(1200 * np.log2(a[:, None] / b[None, :])) <= PERSISTENCE_TOL_CENTS))


def _estimacion_circular(hist, sum_sin, sum_cos, total_w):
    pico_bin = int(np.argmax(hist))
    pico_dev = pico_bin - 49.5
    dev = np.arange(NBINS) - 49.5
    cerca = np.abs(_envolver(dev - pico_dev)) <= 15
    th = 2 * np.pi * dev / 100
    s = float(np.sum(hist[cerca] * np.sin(th[cerca])))
    c = float(np.sum(hist[cerca] * np.cos(th[cerca])))
    offset = math.atan2(s, c) / (2 * math.pi) * 100
    R = math.hypot(sum_sin, sum_cos) / total_w if total_w > 0 else 0.0
    return offset, R


def medir_afinacion(x, sr=SR):
    """x: (canales, N). Devuelve dict con ref_hz, offset (¢ vs A440), unc, R, deriva, segmentos…"""
    largo = x.shape[1]
    usable = largo - FFT_N
    if usable < 0:
        raise ValueError("El audio es demasiado corto para medir la afinación.")
    n_frames = min(MAX_FRAMES, max(8, usable // (FFT_N // 2)))
    hop = usable / max(1, n_frames - 1)
    inicios = [min(usable, int(_redondear_js(f * hop))) for f in range(n_frames)]
    picos = [_picos_ventana(x, ini, sr) for ini in inicios]

    # persistencia (lógica OR con la ventana vecina; red de seguridad si excluiría todo)
    tonal = []
    for f in range(n_frames):
        if picos[f][0].size == 0:
            tonal.append(False)
            continue
        prev = f > 0 and _coinciden(picos[f][2], picos[f - 1][2])
        sig = f < n_frames - 1 and _coinciden(picos[f][2], picos[f + 1][2])
        tonal.append(prev or sig)
    excluidas = tonal.count(False)
    filtrar = excluidas < n_frames
    if not filtrar:
        excluidas = 0

    hist = np.zeros(NBINS)
    sum_sin = sum_cos = total_w = 0.0
    seg = [[0.0, 0.0, 0.0] for _ in range(SEGMENTS)]
    mitad_hist = [np.zeros(NBINS), np.zeros(NBINS)]
    mitad = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    for f in range(n_frames):
        if filtrar and not tonal[f]:
            continue
        dev, w, _ = picos[f]
        if dev.size == 0:
            continue
        s_i = min(SEGMENTS - 1, int(math.floor(inicios[f] / largo * SEGMENTS)))
        h_i = f % 2
        bins = np.clip(_redondear_js(dev + 49.5), 0, NBINS - 1)
        th = 2 * np.pi * dev / 100
        ws, wc, wt = float(np.sum(w * np.sin(th))), float(np.sum(w * np.cos(th))), float(np.sum(w))
        np.add.at(hist, bins, w)
        np.add.at(mitad_hist[h_i], bins, w)
        sum_sin += ws; sum_cos += wc; total_w += wt
        for acc in (seg[s_i], mitad[h_i]):
            acc[0] += ws; acc[1] += wc; acc[2] += wt

    if total_w <= 0:
        raise ValueError("No se encontraron picos tonales para medir la afinación.")
    offset, R = _estimacion_circular(hist, sum_sin, sum_cos, total_w)

    split_diff = None
    if mitad[0][2] > 0 and mitad[1][2] > 0:
        a, _ = _estimacion_circular(mitad_hist[0], *mitad[0])
        b, _ = _estimacion_circular(mitad_hist[1], *mitad[1])
        split_diff = abs(float(_envolver(a - b)))
    disp = math.sqrt(max(0.0, -2 * math.log(max(1e-6, R)))) * 100 / (2 * math.pi)
    unc_analitica = max(0.1, min(25.0, disp / math.sqrt(max(1, n_frames))))
    unc = max(0.1, split_diff) if split_diff is not None else max(unc_analitica, 8.0)

    segmentos = []
    for s, c, wt in seg:
        if wt <= 0:
            segmentos.append(None)
            continue
        d = math.atan2(s, c) / (2 * math.pi) * 100
        segmentos.append(offset + float(_envolver(d - offset)))
    validos = [s for s in segmentos if s is not None]
    deriva = max(abs(s - offset) for s in validos) if validos else 0.0

    return {"ref_hz": 440.0 * 2 ** (offset / 1200), "offset": offset, "R": R, "unc": unc,
            "split_diff": split_diff, "deriva_max": deriva, "segmentos": segmentos,
            "ventanas": n_frames, "excluidas": excluidas}


def frecuencia_dominante(x, sr=SR):
    """Frecuencia absoluta del pico más fuerte (mediana entre ventanas). Solo para autopruebas."""
    usable = x.shape[1] - FFT_N
    n = min(MAX_FRAMES, max(8, usable // (FFT_N // 2)))
    hop = usable / max(1, n - 1)
    fs = []
    for k in range(n):
        _, _, f = _picos_ventana(x, min(usable, int(round(k * hop))), sr)
        if f.size:
            fs.append(f[0])
    return float(np.median(fs))

print("✓ Medición de afinación lista")


## 5 · Rubber Band R3
Mismas opciones que centrail. Autoprueba: +100 ¢ con error ≤ 0,5 ¢; si pylibrb falla, se compila Rubber Band desde el código fuente.

In [ ]:
#@title 5 · Rubber Band (motor fino R3) + autoprueba
# Mismas opciones que centrail (engine/correct.mjs): ENGINE_FINER | PROCESS_OFFLINE |
# PITCH_HIGH_QUALITY. Sin preservar formantes y sin PITCH_HIGH_CONSISTENCY: es una
# corrección de pocos cents sobre la mezcla completa.
# Motor principal: pylibrb (Rubber Band 3 compilado, desde PyPI). Plan B: compilar el
# programa rubberband 3 desde el código fuente y guardarlo en Drive/stems/modelos/bin.
BLOQUE_RB = 65536
RB_VERSION_FUENTE = "3.3.0"
RB_BINARIO = MODELOS / "bin" / f"rubberband-{RB_VERSION_FUENTE}"


def _opciones_pylibrb():
    import pylibrb
    O = pylibrb.Option
    alta_calidad = getattr(O, "PITCH_HIGH_QUALITY", None) or getattr(O, "PitchHighQuality")
    return O.ENGINE_FINER | O.PROCESS_OFFLINE | alta_calidad


def _afinar_pylibrb(x, escala):
    import pylibrb
    canales, n = x.shape
    rb = pylibrb.RubberBandStretcher(sample_rate=SR, channels=canales, options=_opciones_pylibrb(),
                                     initial_time_ratio=1.0, initial_pitch_scale=escala)
    if rb.engine_version != 3:
        raise RuntimeError(f"pylibrb usa el motor {rb.engine_version}, se necesita R3")
    rb.set_expected_input_duration(n)
    rb.set_max_process_size(BLOQUE_RB)
    x = np.ascontiguousarray(x, dtype=np.float32)
    bloques = 2 * math.ceil(n / BLOQUE_RB)  # estudio + proceso, para el avance
    for k, i in enumerate(range(0, n, BLOQUE_RB)):  # 1) estudio (pasada de análisis offline)
        avisar_segmento(k, bloques)
        rb.study(np.ascontiguousarray(x[:, i:i + BLOQUE_RB]), final=i + BLOQUE_RB >= n)
    partes = []
    for k, i in enumerate(range(0, n, BLOQUE_RB)):  # 2) proceso
        avisar_segmento(bloques // 2 + k, bloques)
        rb.process(np.ascontiguousarray(x[:, i:i + BLOQUE_RB]), final=i + BLOQUE_RB >= n)
        partes.append(rb.retrieve_available())
    while not rb.is_done():
        partes.append(rb.retrieve_available())
    y = np.concatenate([p for p in partes if p.size], axis=1)
    return _ajustar_largo(y, n)


def _ajustar_largo(y, n):
    if y.shape[1] >= n:
        return np.ascontiguousarray(y[:, :n], dtype=np.float32)
    return np.pad(y, ((0, 0), (0, n - y.shape[1]))).astype(np.float32)


def compilar_rubberband():
    """Plan B: compila rubberband 3 (programa de línea de comandos) y lo guarda en Drive."""
    if RB_BINARIO.exists():
        return RB_BINARIO
    print("  Compilando Rubber Band", RB_VERSION_FUENTE, "desde el código fuente (~2 min)…")
    subprocess.run(["apt-get", "-qq", "install", "-y", "meson", "ninja-build", "libsndfile1-dev"], check=True)
    carpeta = TRABAJO / "_rubberband_fuente"
    carpeta.mkdir(parents=True, exist_ok=True)
    tar = carpeta / f"rubberband-{RB_VERSION_FUENTE}.tar.bz2"
    subprocess.run(["wget", "-q", "-O", str(tar),
                    f"https://breakfastquay.com/files/releases/rubberband-{RB_VERSION_FUENTE}.tar.bz2"], check=True)
    subprocess.run(["tar", "xjf", str(tar), "-C", str(carpeta)], check=True)
    fuente = carpeta / f"rubberband-{RB_VERSION_FUENTE}"
    subprocess.run(["meson", "setup", "build", "-Dfft=builtin", "-Dresampler=builtin", "-Dcmdline=enabled"],
                   cwd=fuente, check=True, capture_output=True)
    subprocess.run(["ninja", "-C", "build"], cwd=fuente, check=True, capture_output=True)
    RB_BINARIO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fuente / "build" / "rubberband", RB_BINARIO)
    borrar_seguro(carpeta)
    return RB_BINARIO


def _afinar_cli(x, escala):
    binario = compilar_rubberband()
    entrada, salida = TRABAJO / "_rb_entrada.wav", TRABAJO / "_rb_salida.wav"
    sf.write(str(entrada), x.T, SR, subtype="FLOAT")
    subprocess.run([str(binario), "--fine", "--pitch-hq", "-q", "-f", repr(escala), str(entrada), str(salida)],
                   check=True)
    y, _ = sf.read(str(salida), dtype="float32", always_2d=True)
    borrar_seguro(entrada)
    borrar_seguro(salida)
    return _ajustar_largo(y.T, x.shape[1])


MOTOR_RB = None  # lo fija la autoprueba: "pylibrb" o "cli"


def afinar(x, cents):
    """Pitch-shift de `cents` sin cambiar el tempo, con el motor que pasó la autoprueba."""
    escala = 2 ** (cents / 1200)
    return _afinar_pylibrb(x, escala) if MOTOR_RB == "pylibrb" else _afinar_cli(x, escala)


def _tono_prueba(f0, segundos=8.0):
    t = np.arange(int(segundos * SR)) / SR
    y = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.1 * np.sin(2 * np.pi * 2 * f0 * t) + 0.05 * np.sin(2 * np.pi * 3 * f0 * t)
    return np.stack([y, y]).astype(np.float32)


def autoprueba_rubberband(motor):
    """Tono de 440 Hz desplazado +100 ¢: la frecuencia medida debe quedar a ≤ 0,5 ¢ de 466,16 Hz."""
    global MOTOR_RB
    MOTOR_RB = motor
    x = _tono_prueba(440.0)
    y = afinar(x, 100.0)
    esperado = 440.0 * 2 ** (100 / 1200)
    error = 1200 * math.log2(frecuencia_dominante(y) / esperado)
    return error, y.shape == x.shape


for _motor in ("pylibrb", "cli"):
    try:
        _err, _largo_ok = autoprueba_rubberband(_motor)
        if abs(_err) <= 0.5 and _largo_ok:
            print(f"✓ Rubber Band R3 vía {_motor}: autoprueba +100 ¢ con error {_err:+.3f} ¢")
            break
        print(f"⚠ {_motor}: autoprueba fallida (error {_err:+.3f} ¢, largo correcto: {_largo_ok})")
    except Exception as e:
        print(f"⚠ {_motor} no disponible: {e}")
    MOTOR_RB = None
if MOTOR_RB is None and AFINAR:
    raise Detenido("Ningún motor de Rubber Band pasó la autoprueba: no se puede afinar. "
                   "Desmarca AFINAR en la configuración para separar sin afinar, o vuelve a intentar más tarde.")


## 6 · Modelos
Catálogo de audio-separator (resuelto con su lista oficial), modelos de guitarra desde HuggingFace (`noblebarkrr/mvsepless_resources`, rutas explícitas) y cuatro arreglos en memoria: `mlp_expansion_factor`, carga fuera del catálogo, modelos de más de 16 stems y avance de Demucs por segmento.

In [ ]:
#@title 6 · Modelos: catálogo, guitarras desde HuggingFace y arreglos de audio-separator
import difflib, gc, logging
from importlib.metadata import version as _version

if _version("audio-separator") != "0.44.5":
    raise Detenido(f"Se esperaba audio-separator 0.44.5 y hay {_version('audio-separator')}. "
                   "El arreglo de MelBandRoformer depende de esa versión: reinstala 0.44.5.")

from audio_separator.separator import Separator
import audio_separator.separator.uvr_lib_v5.roformer.mel_band_roformer as _mbr

# ---------- Arreglo 1 (monkeypatch, solo en memoria): mlp_expansion_factor ----------
# En 0.44.5, MelBandRoformer.__init__ recibe mlp_expansion_factor pero NO se lo pasa a su
# MaskEstimator, que usa siempre 4. Eso rompe becruily (factor 1) y listra92 (factor 2).
# BS-Roformer no tiene el bug. No se toca site-packages.
if not getattr(_mbr.MelBandRoformer, "_arreglo_mlp", False):
    _MaskEstimatorOriginal = _mbr.MaskEstimator
    _factor_actual = [4]

    class _MaskEstimatorArreglado(_MaskEstimatorOriginal):
        def __init__(self, dim, dim_inputs, depth, mlp_expansion_factor=None):
            if mlp_expansion_factor is None:
                mlp_expansion_factor = _factor_actual[0]
            super().__init__(dim=dim, dim_inputs=dim_inputs, depth=depth, mlp_expansion_factor=mlp_expansion_factor)

    _init_original = _mbr.MelBandRoformer.__init__

    def _init_arreglado(self, *args, mlp_expansion_factor=4, **kwargs):
        _factor_actual[0] = mlp_expansion_factor
        try:
            _init_original(self, *args, mlp_expansion_factor=mlp_expansion_factor, **kwargs)
        finally:
            _factor_actual[0] = 4

    _mbr.MaskEstimator = _MaskEstimatorArreglado
    _mbr.MelBandRoformer.__init__ = _init_arreglado
    _mbr.MelBandRoformer._arreglo_mlp = True

# ---------- Arreglo 2: cargar checkpoints que no están en el catálogo ----------
# El CLI (-m) los rechaza aunque el archivo exista. Se usa la API interna y, para los
# modelos propios, se salta la búsqueda en el catálogo devolviendo la ruta local.
MODELOS_PROPIOS = {}  # nombre_archivo.ckpt -> nombre_archivo.yaml
if not getattr(Separator, "_arreglo_catalogo", False):
    _descarga_original = Separator.download_model_files

    def _descarga_arreglada(self, model_filename):
        if model_filename in MODELOS_PROPIOS:
            ruta = os.path.join(self.model_file_dir, model_filename)
            return model_filename, "MDXC", Path(model_filename).stem, ruta, MODELOS_PROPIOS[model_filename]
        return _descarga_original(self, model_filename)

    Separator.download_model_files = _descarga_arreglada
    Separator._arreglo_catalogo = True

# ---------- Arreglo 3: modelos con más de 16 stems ----------
# El validador de 0.44.5 limita num_stems a 1–16 y rechaza el BS-Roformer "mega" de
# 53 stems (usado para la guitarra acústica). Se amplía el rango solo en memoria.
from audio_separator.separator.roformer.parameter_validator import ParameterValidator as _Validador
if _Validador.PARAMETER_RANGES.get("num_stems", (1, 64))[1] < 64:
    _Validador.PARAMETER_RANGES = {**_Validador.PARAMETER_RANGES, "num_stems": (1, 64)}

# ---------- Arreglo 4 (solo visual): avance de Demucs por segmento ----------
# Demucs dibuja su propia barra tqdm por cada pasada; se reemplaza en memoria por un
# envoltorio que informa "segmento k de M" al avance del notebook, sin imprimir líneas.
import types as _types
import audio_separator.separator.uvr_lib_v5.demucs.apply as _demucs_apply


def _iterar_con_avance(iterable, *args, **kwargs):
    elementos = list(iterable)
    for k, e in enumerate(elementos):
        avisar_segmento(k, len(elementos))
        yield e
    avisar_segmento(len(elementos), len(elementos))


_demucs_apply.tqdm = _types.SimpleNamespace(tqdm=_iterar_con_avance)

# ---------- Modelos del catálogo (resueltos con la lista oficial, nunca de memoria) ----------
_sep_info = Separator(info_only=True, log_level=logging.WARNING, model_file_dir=str(MODELOS))
_catalogo = set()
for _tipo, _modelos in _sep_info.list_supported_model_files().items():
    for _nombre, _info in _modelos.items():
        _catalogo.add(_info["filename"])
        _catalogo.update(f.split("/")[-1] for f in _info.get("download_files", []))


def resolver_catalogo(que, candidatos):
    for c in candidatos:
        if c in _catalogo:
            return c
    parecidos = difflib.get_close_matches(candidatos[0], sorted(_catalogo), n=8, cutoff=0.3)
    raise Detenido(f"No encontré el modelo de {que} en el catálogo de audio-separator 0.44.5.\n"
                   f"Busqué: {candidatos}\nParecidos: {parecidos}")


MODELO_VOZ = resolver_catalogo("voz (BS-RoFormer)", [
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt", "model_bs_roformer_ep_368_sdr_12.9628.ckpt"])
MODELO_KARAOKE = resolver_catalogo("karaoke (Mel-Band RoFormer)", [
    "mel_band_roformer_karaoke_becruily.ckpt", "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
    "mel_band_roformer_karaoke_gabox.ckpt"])
MODELO_DEMUCS = resolver_catalogo("Demucs 6 stems", ["htdemucs_6s.yaml"])

# ---------- Modelos de guitarra (comunidad MSST, HuggingFace) ----------
# Rutas EXPLÍCITAS verificadas en el repo (no hay models.json). El nombre local en
# Drive/stems/modelos contiene "roformer" y el tipo (mel_band_ / bs_): audio-separator
# detecta la arquitectura por el nombre del archivo.
REPO_HF = "noblebarkrr/mvsepless_resources"
MODELOS_HF = {  # clave: (ckpt en el repo, yaml en el repo, nombre local sin extensión)
    "becruily": ("mel_band_roformer/mbr_guitar_becruily.ckpt",
                 "mel_band_roformer/mbr_guitar_becruily_config.yaml",
                 "mel_band_roformer_guitar_becruily"),
    "chencfd": ("mel_band_roformer/mbr_guitar_chencfd.ckpt",
                "mel_band_roformer/mbr_guitar_chencfd_config.yaml",
                "mel_band_roformer_guitar_chencfd"),
    "punteo": ("mel_band_roformer/mbr_lead_rhythm_guitar_listra92.ckpt",
               "mel_band_roformer/mbr_lead_rhythm_guitar_listra92_config.yaml",
               "mel_band_roformer_lead_rhythm_guitar_listra92"),
    # acústica: BS-Roformer "mega" de MVSep, 53 stems (1,37 GB). Solo se usa su stem
    # 'acoustic-guitar'; las otras 52 salidas se descartan sin guardarlas.
    "acustica": ("bs_roformer/bs_mega_53stem_full_mvsep.ckpt",
                 "bs_roformer/bs_mega_53stem_full_mvsep_config.yaml",
                 "bs_roformer_mega_53stem_mvsep"),
}
_archivos_repo = []


def preparar_modelo_hf(que):
    """Descarga (una sola vez, a Drive/stems/modelos) y registra un modelo de HuggingFace."""
    ckpt_repo, yaml_repo, local = MODELOS_HF[que]
    ckpt_local, yaml_local = MODELOS / f"{local}.ckpt", MODELOS / f"{local}.yaml"
    if not (ckpt_local.exists() and yaml_local.exists()):
        from huggingface_hub import hf_hub_download, list_repo_files
        if not _archivos_repo:
            _archivos_repo.extend(list_repo_files(REPO_HF))
        faltan = [r for r in (ckpt_repo, yaml_repo) if r not in _archivos_repo]
        if faltan:
            con_guitar = [f for f in _archivos_repo if "guitar" in f.lower()]
            raise Detenido(f"No están en {REPO_HF}: {faltan}\nArchivos del repo con 'guitar':\n  "
                           + "\n  ".join(con_guitar))
        descargas = TRABAJO / "_descargas_hf"
        for repo, destino in ((yaml_repo, yaml_local), (ckpt_repo, ckpt_local)):
            print(f"    descargando {repo} → {destino.name}")
            temporal = hf_hub_download(REPO_HF, repo, local_dir=str(descargas))
            parcial = destino.with_name(destino.name + ".parcial")
            shutil.copyfile(temporal, parcial)
            os.replace(parcial, destino)
        borrar_seguro(descargas)  # copia temporal local (dentro de /content/trabajo)
    MODELOS_PROPIOS[ckpt_local.name] = yaml_local.name
    return ckpt_local.name


MODELO_GUITARRAS = MODELO_PUNTEO = MODELO_ACUSTICA = None
if MODO == "completo" and ACCION == "procesar todo":  # en «solo clic» no se descarga nada de guitarras
    MODELO_GUITARRAS = preparar_modelo_hf(MODELO_GUITARRA)
    MODELO_PUNTEO = preparar_modelo_hf("punteo")
    if SEPARAR_ACUSTICA_ELECTRICA:
        MODELO_ACUSTICA = preparar_modelo_hf("acustica")

print("✓ Modelos:")
for _k, _v in [("voz", MODELO_VOZ), ("karaoke", MODELO_KARAOKE), ("demucs", MODELO_DEMUCS),
               ("guitarras", MODELO_GUITARRAS), ("punteo", MODELO_PUNTEO), ("acústica", MODELO_ACUSTICA)]:
    if _v:
        print(f"   {_k:10s} {_v}")


## 7 · Separación en cascada
Cada etapa: extraído + resto (= entrada − extraído). Los RoFormer corren con un bucle propio que guarda solo los stems pedidos (el modelo de 53 stems no cabe completo en la RAM).

In [ ]:
#@title 7 · Separación en cascada con restas exactas
# PRINCIPIO: cada etapa entrega "lo extraído" (salida del modelo) y "el resto" =
# entrada − extraído, por RESTA EXACTA en float32 sobre la misma señal. Nunca se usa una
# segunda salida independiente del modelo. Así la suma de todos los stems = la mezcla.
#
# Para no romper la resta, los modelos se llaman directo, sin el camino normal de
# audio-separator (separate), que normaliza entrada y salidas y las cuantiza al escribir.
# RoFormer: bucle de inferencia propio por segmentos (ventana de Hamming + solape), que
# guarda SOLO los stems pedidos: el modelo "mega" de 53 stems no cabe en la RAM de Colab
# si se acumulan todas sus salidas. Si la GPU se queda sin memoria, se reintenta con
# segmentos de la mitad (mínimo SEGMENTO_MIN_S) y el valor usado queda en el manifiesto.
SOLAPAMIENTO_MAX = 4      # tope de pasos por segmento (num_overlap del yaml, si es menor)
SEGMENTO_MIN_S = 2.0

# Qué salida del modelo es "lo extraído" en cada etapa (palabras clave del nombre del stem).
# Si el modelo solo entrega el complemento, extraído = entrada − complemento (sigue exacto).
# "exacto": nombre exacto del stem (para modelos con muchos stems parecidos).
ETAPAS = {
    "voz":           {"objetivo": ["vocal"], "complemento": ["instrument", "other"]},
    "voz_principal": {"objetivo": ["lead", "vocal"], "complemento": ["karaoke", "back", "instrument", "other"]},
    "guitarras":     {"objetivo": ["guitar"], "complemento": ["other", "rest", "instrument"]},
    "punteo":        {"objetivo": ["lead"], "complemento": ["rhythm", "other"]},
    # modelo mega de 53 stems: solo 'acoustic-guitar' (no 'guitar' ni 'electric-guitar')
    "acustica":      {"exacto": "acoustic-guitar"},
}
NOMBRES_ETAPA = {"voz": "voz (BS-RoFormer)", "voz_principal": "voz principal / coros (karaoke)",
                 "guitarras": "guitarras", "punteo": "punteo / rítmica (listra92)",
                 "acustica": "acústica / eléctrica (BS-Roformer mega)",
                 "demucs": "batería, bajo, piano (Demucs 6 stems)"}
_EXCLUYE = ["no_", "no-", "without", "other", "rest", "instrument", "karaoke", "back", "rhythm", "electric"]


def elegir_salida(salidas, entrada, etapa):
    """Devuelve (extraído, descripción) a partir del dict de salidas del modelo."""
    reglas = ETAPAS[etapa]
    if "exacto" in reglas:
        if reglas["exacto"] not in salidas:
            raise RuntimeError(f"Etapa {etapa}: el modelo no entregó '{reglas['exacto']}' (entregó {sorted(salidas)})")
        return salidas[reglas["exacto"]], f"salida '{reglas['exacto']}'"
    for palabra in reglas["objetivo"]:
        for k, v in salidas.items():
            if palabra in k and not any(e in k for e in _EXCLUYE if e != palabra):
                return v, f"salida '{k}'"
    for palabra in reglas["complemento"]:
        for k, v in salidas.items():
            if palabra in k:
                return entrada - v, f"entrada − salida '{k}'"
    raise RuntimeError(f"Etapa {etapa}: no reconozco las salidas del modelo {list(salidas)}")


def _igualar_largo(y, n):
    y = np.asarray(y, dtype=np.float32)
    if y.shape[-1] >= n:
        return np.ascontiguousarray(y[..., :n])
    return np.pad(y, ((0, 0), (0, n - y.shape[-1])))


def cargar_modelo(archivo):
    sep = Separator(log_level=logging.WARNING, model_file_dir=str(MODELOS),
                    output_dir=str(TRABAJO / "_separador"), sample_rate=SR)
    try:
        sep.load_model(archivo)
    except SystemExit:  # audio-separator llama sys.exit(1) si falla la carga: no detener todo
        raise RuntimeError(f"audio-separator no pudo cargar {archivo} (ver mensajes arriba)")
    return sep


def liberar_modelo(sep):
    sep.model_instance = None
    del sep
    gc.collect()
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def nombres_stems(modelo, datos):
    """Nombres de las salidas del RoFormer, en el orden en que las entrega."""
    entrenamiento = datos.get("training", {})
    instrumentos = list(entrenamiento.get("instruments") or [])
    objetivo = entrenamiento.get("target_instrument")
    n = int(getattr(modelo, "num_stems", len(instrumentos) or 1))
    if n > 1:
        if len(instrumentos) != n:
            raise RuntimeError(f"El yaml declara {len(instrumentos)} instrumentos y el modelo entrega {n}")
        return instrumentos
    return [objetivo or (instrumentos[0] if instrumentos else "salida")]


def parametros_segmento(datos):
    """(muestras por segmento, pasos de solape) según el yaml del modelo.
    Prioridad: inference.chunk_size (lo que el autor fijó para separar; en el mega 882000 = 20 s)
    > audio.chunk_size (largo de entrenamiento; en el mega 441000) > hop × (dim_t − 1)."""
    audio, inferencia, modelo = datos.get("audio", {}), datos.get("inference", {}), datos.get("model", {})
    seg = inferencia.get("chunk_size") or audio.get("chunk_size")
    if not seg:
        hop = modelo.get("stft_hop_length") or audio.get("hop_length") or 441
        seg = int(hop) * (int(inferencia.get("dim_t") or audio.get("dim_t") or 801) - 1)
    solape = min(SOLAPAMIENTO_MAX, max(1, int(inferencia.get("num_overlap") or SOLAPAMIENTO_MAX)))
    return int(seg), solape


def _inferir_segmentos(modelo, x, indices, seg, solape, dispositivo):
    import torch
    n = x.shape[1]
    X = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))
    # Algunos RoFormer devuelven un segmento algo más corto que la entrada (hop del STFT):
    # se suma solo lo devuelto y se rellena el final con ceros para que toda muestra real
    # quede cubierta por al menos un segmento.
    relleno = max(seg - n, 0) + 8192
    X = torch.nn.functional.pad(X, (0, relleno))
    largo = X.shape[1]
    paso = max(1, seg // solape)
    inicios = list(range(0, largo - seg + 1, paso))
    if inicios[-1] != largo - seg:
        inicios.append(largo - seg)
    ventana = torch.hamming_window(seg, periodic=False)
    acum = torch.zeros((len(indices), 2, largo))
    peso = torch.zeros(largo)
    sel = torch.tensor(indices)
    with torch.no_grad():
        for j, i in enumerate(inicios):
            avisar_segmento(j, len(inicios))
            y = modelo(X[:, i:i + seg].unsqueeze(0).to(dispositivo))[0]
            if y.ndim == 2:
                y = y.unsqueeze(0)
            m = min(seg, y.shape[-1])
            acum[..., i:i + m] += y.index_select(0, sel.to(y.device))[..., :m].float().cpu() * ventana[:m]
            peso[i:i + m] += ventana[:m]
            del y
    avisar_segmento(len(inicios), len(inicios))
    if float(peso[:n].min()) <= 0:
        raise RuntimeError("Inferencia por segmentos: quedaron muestras sin cubrir")
    return (acum[..., :n] / peso[:n]).numpy()


def inferir_roformer(inst, x, solo=None, info_segmento=None):
    """Corre un RoFormer cargado por audio-separator sobre x (2, N). Devuelve {nombre: (2, N)}
    solo con los stems de `solo` (o todos). Sin normalizar niveles."""
    import torch
    datos = inst.model_data
    modelo = inst.model_run
    nombres = [str(k) for k in nombres_stems(modelo, datos)]
    pedidos = nombres if solo is None else list(solo)
    faltan = [p for p in pedidos if p not in nombres]
    if faltan:
        raise RuntimeError(f"El modelo no tiene {faltan}; tiene {nombres}")
    indices = [nombres.index(p) for p in pedidos]
    seg, solape = parametros_segmento(datos)
    dispositivo = next(modelo.parameters()).device
    normalizar = bool(datos.get("inference", {}).get("normalize", False))
    media, desv = (float(x.mean()), float(x.std()) or 1.0) if normalizar else (0.0, 1.0)
    entrada = (x - media) / desv if normalizar else x
    while True:
        try:
            salida = _inferir_segmentos(modelo, entrada, indices, seg, solape, dispositivo)
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if seg // 2 < SEGMENTO_MIN_S * SR:
                raise
            seg //= 2
            print(f"    (memoria de GPU insuficiente: reintento con segmentos de {seg / SR:.1f} s)")
    if info_segmento is not None:
        info_segmento.update(muestras=seg, segundos=round(seg / SR, 2), solape=solape)
    if normalizar:
        salida = salida * desv + media
    return {p.lower(): _igualar_largo(salida[k], x.shape[1]) for k, p in enumerate(pedidos)}


def salidas_modelo(sep, x, solo=None, info_segmento=None):
    """Corre el modelo sobre x (2, N) float32 y devuelve {nombre_en_minúsculas: (2, N)} sin tocar niveles."""
    inst = sep.model_instance
    n = x.shape[1]
    if type(inst).__name__ == "DemucsSeparator":
        from audio_separator.separator.architectures import demucs_separator as ds
        ruta = Path(inst.model_path)
        modelo = ds.get_demucs_model(name=ruta.stem, repo=ruta.parent)
        modelo = ds.demucs_segments(inst.segment_size, modelo)
        modelo.to(inst.torch_device).eval()
        inst.demucs_model_instance = modelo
        try:
            fuentes = inst.demix_demucs(x)
        finally:
            inst.demucs_model_instance = None
            del modelo
        mapa = ds.DEMUCS_6_SOURCE_MAPPER if len(fuentes) == 6 else ds.DEMUCS_4_SOURCE_MAPPER
        return {str(nombre).lower(): _igualar_largo(fuentes[i], n) for nombre, i in mapa.items()}
    if getattr(inst, "is_roformer", False):
        return inferir_roformer(inst, x, solo, info_segmento)
    salida = inst.demix(mix=x)
    if not isinstance(salida, dict):
        salida = {str(inst.primary_stem_name): salida}
    return {str(k).lower(): _igualar_largo(v, n) for k, v in salida.items()}


def separador_real(registro, tiempos, segmentos=None):
    """Devuelve separar(etapa, x) usando los modelos reales. Anota en `registro` qué salida usó
    y en `segmentos` el tamaño de segmento usado por cada RoFormer."""
    modelos = {"voz": MODELO_VOZ, "voz_principal": MODELO_KARAOKE, "guitarras": MODELO_GUITARRAS,
               "punteo": MODELO_PUNTEO, "acustica": MODELO_ACUSTICA, "demucs": MODELO_DEMUCS}

    def separar(etapa, x):
        solo = [ETAPAS[etapa]["exacto"]] if "exacto" in ETAPAS.get(etapa, {}) else None
        info_seg = {}
        with Etapa(NOMBRES_ETAPA[etapa], tiempos):
            sep = cargar_modelo(modelos[etapa])
            try:
                salidas = salidas_modelo(sep, x, solo, info_seg)
            finally:
                liberar_modelo(sep)
        if segmentos is not None and info_seg:
            segmentos[etapa] = info_seg
        if etapa == "demucs":
            registro[etapa] = f"{modelos[etapa]}: {sorted(salidas)}"
            return salidas
        extraido, como = elegir_salida(salidas, x, etapa)
        seg_txt = f", segmento {info_seg['segundos']} s" if info_seg else ""
        registro[etapa] = f"{modelos[etapa]}: {como} (salidas usadas: {sorted(salidas)}{seg_txt})"
        return extraido

    return separar


def cascada(mezcla, separar, modo, orden, por_timbre):
    """Separa `mezcla` (2, N) float32. Devuelve (stems, guitarras_partes, guitarras_total).
    stems: dict "NN Nombre" -> array; su suma es exactamente `mezcla` (salvo redondeo float32)."""
    voz = separar("voz", mezcla)
    instrumental = mezcla - voz
    principal = separar("voz_principal", voz)
    coros = voz - principal
    if modo == "karaoke":
        return {"01 Voz principal": principal, "02 Coros": coros, "03 Instrumental": instrumental}, None, None

    guitarras = separar("guitarras", instrumental)
    sin_guitarras = instrumental - guitarras

    if orden == "punteo primero" or not por_timbre:
        punteo = separar("punteo", guitarras)
        ritmicas = guitarras - punteo
        if por_timbre:
            ritm_ac = separar("acustica", ritmicas)
            ritm_el = ritmicas - ritm_ac
            partes = {"05 Guitarra punteo": punteo, "06 Guitarras ritmicas electricas": ritm_el,
                      "07 Guitarras ritmicas acusticas": ritm_ac}
        else:
            partes = {"05 Guitarra punteo": punteo, "06 Guitarras ritmicas": ritmicas}
    else:  # "timbre primero"
        acusticas = separar("acustica", guitarras)
        electricas = guitarras - acusticas
        punteo = separar("punteo", electricas)
        electricas_resto = electricas - punteo
        partes = {"05 Guitarra punteo": punteo, "06 Guitarras electricas": electricas_resto,
                  "07 Guitarras acusticas": acusticas}

    d = separar("demucs", sin_guitarras)
    otros = sin_guitarras - (d["drums"] + d["bass"] + d["piano"])
    stems = {"01 Voz principal": principal, "02 Coros": coros, "03 Bateria": d["drums"], "04 Bajo": d["bass"]}
    stems.update(partes)
    stems["08 Piano"] = d["piano"]
    stems["09 Otros"] = otros
    return stems, partes, guitarras

print("✓ Separación lista")


## 8 · Informes
`afinacion.txt` (con la prueba de cancelación) y `guitarras.txt` (mapa de punteo).

In [ ]:
#@title 8 · Informes: afinacion.txt y guitarras.txt
# --- Mapa de punteo (guitarras.txt): constantes a la vista para ajustarlas ---
VENTANA_RMS_S = 0.5          # largo de cada ventana de energía
HOP_RMS_S = 0.25             # avance entre ventanas
UMBRAL_PUNTEO_REL = 0.25     # punteo activo si su RMS ≥ 0,25 × RMS de guitarras (−12 dB)
PISO_GUITARRAS_DBFS = -50.0  # y las guitarras suenan sobre −50 dBFS
MIN_TRAMO_S = 2.0            # tramo mínimo
UNIR_HUECO_S = 1.0           # se unen tramos separados por menos de esto
RESIDUO_MAX_DBFS = -90.0     # prueba de cancelación (FLAC 24-bit)


def _rms_ventanas(x, sr):
    e = np.mean(np.asarray(x, dtype=np.float64) ** 2, axis=0)
    v, h = int(VENTANA_RMS_S * sr), int(HOP_RMS_S * sr)
    acum = np.concatenate([[0.0], np.cumsum(e)])
    inicios = np.arange(0, max(1, len(e) - v + 1), h)
    return inicios / sr, np.sqrt(np.maximum(acum[inicios + v] - acum[inicios], 0) / v)


def tramos_punteo(punteo, guitarras, sr=SR):
    """Tramos (inicio_s, fin_s) con punteo sostenido."""
    t, rms_p = _rms_ventanas(punteo, sr)
    _, rms_g = _rms_ventanas(guitarras, sr)
    piso = 10 ** (PISO_GUITARRAS_DBFS / 20)
    activo = (rms_p >= UMBRAL_PUNTEO_REL * rms_g) & (rms_g > piso)
    tramos = []
    for i in np.flatnonzero(activo):
        ini, fin = t[i], t[i] + VENTANA_RMS_S
        if tramos and ini - tramos[-1][1] < UNIR_HUECO_S:
            tramos[-1][1] = max(tramos[-1][1], fin)
        else:
            tramos.append([ini, fin])
    dur = punteo.shape[1] / sr
    return [(a, min(b, dur)) for a, b in tramos if min(b, dur) - a >= MIN_TRAMO_S]


def proporciones(partes, desde=0, hasta=None):
    """Proporción de energía de cada parte de guitarra (suma 100 %) en el rango de muestras."""
    e = {k: float(np.sum(np.asarray(v[:, desde:hasta], dtype=np.float64) ** 2)) for k, v in partes.items()}
    total = sum(e.values())
    return {k: (100 * v / total if total > 0 else 0.0) for k, v in e.items()}


def texto_guitarras(tema, partes, guitarras, orden, sr=SR):
    nombres = {k: k[3:] for k in partes}
    tramos = tramos_punteo(partes["05 Guitarra punteo"], guitarras, sr)
    lineas = [f"GUITARRAS — {tema}", "",
              f"Orden usado: {orden}" + ("" if len(partes) > 2 else " (sin corte acústica/eléctrica)"), "",
              "Proporción de energía de guitarras en todo el tema:"]
    for k, v in proporciones(partes).items():
        lineas.append(f"  {nombres[k]:32s} {v:5.1f} %")
    lineas += ["", f"Tramos con punteo sostenido ({len(tramos)}):",
               f"  (punteo ≥ {UMBRAL_PUNTEO_REL:g} × guitarras, guitarras > {PISO_GUITARRAS_DBFS:g} dBFS, "
               f"mínimo {MIN_TRAMO_S:g} s, huecos < {UNIR_HUECO_S:g} s unidos)"]
    if not tramos:
        lineas.append("  ninguno")
    for a, b in tramos:
        p = proporciones(partes, int(a * sr), int(b * sr))
        detalle = " · ".join(f"{nombres[k]} {v:.0f} %" for k, v in p.items())
        lineas.append(f"  {mm_ss(a):>7s} – {mm_ss(b):>7s}  ({b - a:4.1f} s)  {detalle}")
    lineas += ["", "Sirve para ver rápido si detectó el solo real o se llevó un riff.",
               "Nada de esto vale hasta escucharlo contra la canción."]
    return "\n".join(lineas) + "\n"


def _nivel_incertidumbre(unc):
    return "alta repetibilidad" if unc <= 1.5 else ("media" if unc <= 6 else "variación alta del material")


def _lineas_medicion(titulo, m, destino):
    dev_dest = 1200 * math.log2(m["ref_hz"] / destino)
    L = [titulo,
         f"  Referencia detectada:        {m['ref_hz']:.2f} Hz",
         f"  Desviación respecto de A440: {m['offset']:+.2f} ¢",
         f"  Desviación respecto destino: {dev_dest:+.2f} ¢",
         f"  Incertidumbre:               ±{m['unc']:.2f} ¢ ({_nivel_incertidumbre(m['unc'])}; ventanas pares vs. impares)",
         f"  Consistencia tonal (R):      {100 * m['R']:.1f} % (informativo)",
         f"  Deriva máxima:               {m['deriva_max']:.2f} ¢ (10 segmentos)",
         "  Segmentos (¢ vs A440):       " + " ".join("—" if s is None else f"{s:+.1f}" for s in m["segmentos"]),
         "  (los segmentos son la media de TODOS los picos de cada tramo; el valor global usa el pico del",
         "   histograma ±15 ¢, igual que centrail: por eso pueden no coincidir)",
         f"  Ventanas analizadas:         {m['ventanas']} (excluidas sin contenido tonal: {m['excluidas']})"]
    if m["deriva_max"] > 10:
        L.append("  ⚠ ADVERTENCIA: la deriva supera 10 ¢; una corrección global no deja todo el tema afinado.")
    return L


def texto_afinacion(tema, info):
    L = [f"AFINACIÓN — {tema}", ""]
    L.append(f"Entrada: {info['archivo']} (frecuencia nativa {info['sr_nativa']} Hz; se trabaja a 44100 Hz)")
    dest = info["destino_hz"]
    for a in info.get("avisos_manual", []):
        L.append(f"⚠ {tema}.afinacion.txt: {a}")
    if dest is None and not info.get("manual"):
        L += ["Afinación: no se midió ni se corrigió (AFINAR desactivado)."]
    else:
        if dest is not None:
            L += [f"Destino: {dest:.2f} Hz · umbral ±{UMBRAL_CENTS:g} ¢", ""]
        if info.get("medicion"):
            L += _lineas_medicion("Medición 1 — mezcla original − batería (Demucs), método de centrail:",
                                  info["medicion"], dest)
            L.append(f"  Confianza: {info.get('nivel_medicion', '—')} → corrección indicada {info['shift_medicion1']:+.2f} ¢")
            L.append("")
        if info.get("medicion2"):
            L += _lineas_medicion("Medición 2 — lazo cerrado: mezcla ya corregida − batería:", info["medicion2"], dest)
            L.append(f"  Residuo respecto del destino: {info['residuo2']:+.2f} ¢")
            L.append("")
        if info["corregido"]:
            L.append(f"VALOR APLICADO: {info['shift']:+.2f} ¢ · fuente: {info['fuente']}")
            L.append(f"  Por qué: {info['razon']}.")
            L.append(f"  Rubber Band R3 ({info['motor']}), UNA vez sobre la mezcla original; todo lo demás sale "
                     "de la mezcla afinada.")
        else:
            L.append(f"VALOR APLICADO: ninguno · fuente: {info['fuente']}")
            L.append(f"  Por qué: {info['razon'] or 'sin corrección'}.")
        if info.get("ganancia_afinar_db", 0.0) < -0.01:
            L.append(f"Nivel: se redujo {-info['ganancia_afinar_db']:.2f} dB tras afinar para no superar "
                     f"{'0 dBFS' if info['pico_original'] >= 1 else 'el nivel del original'}.")
        for a in info.get("avisos_afinacion", []):
            L.append(a)
        v = info.get("verificacion")
        if v:
            L.append("")
            L.append(f"Verificación tras separar (mezcla afinada − batería): {v['ref_hz']:.2f} Hz, "
                     f"{info['verif_cents']:+.2f} ¢ respecto del destino (±{v['unc']:.2f} ¢).")
            if info.get("verif_fuera"):
                m1 = info.get("medicion")
                L.append(f"⚠ ADVERTENCIA: tras corregir, la afinación quedó a {info['verif_cents']:+.2f} ¢ del destino "
                         f"(umbral ±{UMBRAL_CENTS:g} ¢).")
                L.append("  " + (f"La medición inicial era incierta (±{m1['unc']:.2f} ¢). " if m1 else "")
                         + f"Escucha antes de usarlo; si hace falta, corrige a mano con {tema}.afinacion.txt "
                         "(p. ej. «corregir -8.0») y agrega el tema a REPROCESAR.")
    L += ["", "Prueba de cancelación (mezcla de referencia − suma de stems, releyendo los archivos escritos):",
          f"  Referencia: {'00 Mezcla afinada' if info['corregido'] else 'mezcla original'}"
          f" · {FORMATO} · {SR_SALIDA} Hz"]
    if info.get("ganancia_recorte_db", 0.0) < -0.01:
        L.append(f"  Nota: para evitar recorte se bajaron {-info['ganancia_recorte_db']:.2f} dB por igual a todos los "
                 "stems y a la referencia (la suma sigue igual a la mezcla × esa ganancia).")
    L.append(f"  Residuo pico: {info['residuo_dbfs']:.1f} dBFS → {info['veredicto_cancelacion']}")
    L += ["", "Modelos y salidas usadas:"] + [f"  {k}: {v}" for k, v in info["modelos"].items()]
    L += ["", "Tiempos (s): " + ", ".join(f"{k} {v:g}" for k, v in info["tiempos"].items())]
    return "\n".join(L) + "\n"

print("✓ Informes listos")


## 8b · Clic
Pulsos y downbeats con [Beat This!](https://github.com/CPJKU/beat_this) sobre la mezcla afinada (en CPU si no hay GPU). Escribe `<tema>/clic/`: `clic.flac` (mismo largo y frecuencia que los stems, desfase 0), `clic_marcas.csv`, `clic.mid` (mapa de tempo para Reaper), `clic_marcadores_reaper.csv` y `clic_informe.txt` (zonas dudosas y cómo importarlo). Si existe `<tema>.clic.csv` junto al audio (exportado de Reaper), se usa en vez de la detección.

In [ ]:
#@title 8b · Clic: pulsos con Beat This!, clic.flac, marcas, MIDI y marcadores para Reaper
# El clic NO es un stem: va en <tema>/clic/ y no entra en la prueba de cancelación.
# Corre en CPU si no hay GPU (Beat This! es un modelo chico).
import csv, re
BEAT_THIS_URL = "https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt"
BEAT_THIS_FPS = 50                 # Beat This! entrega pulsos en cuadros de 20 ms (±10 ms)
CLIC_TONO_DOWNBEAT, CLIC_TONO_PULSO = 1568.0, 1047.0   # Hz: Sol6 en el 1 de cada compás, Do6 en el resto
CLIC_NIVEL_DOWNBEAT, CLIC_NIVEL_PULSO = 0.9, 0.6
CLIC_DURACION_S = 0.025
# zonas dudosas (para clic_informe.txt)
DUDA_CAMBIO_BRUSCO = 0.08          # un pulso que se aparta > 8 % de la mediana local
DUDA_RUBATO_CV = 0.03              # el intervalo cambia > 3 % de pulso a pulso (promedio en 8 pulsos)
DUDA_CONFIANZA = 0.5               # activación de Beat This! bajo 0,5
DUDA_HUECO = 1.8                   # intervalo > 1,8 × el típico = pulsos que faltan
SIN_BATERIA_DB = -25.0             # batería 25 dB bajo su nivel típico, al menos 4 s
MIDI_PPQ = 960
_beat_this = {}


def _cargar_beat_this():
    """Descarga (una vez, a la carpeta de modelos) y carga Beat This! en GPU si hay, si no en CPU."""
    if "modelo" not in _beat_this:
        import torch
        from beat_this.inference import Audio2Frames
        from beat_this.model.postprocessor import Postprocessor
        ckpt = MODELOS / "beat_this_final0.ckpt"
        if not ckpt.exists():
            parcial = ckpt.with_name(ckpt.name + ".parcial")
            torch.hub.download_url_to_file(BEAT_THIS_URL, str(parcial), progress=False)
            os.replace(parcial, ckpt)
        dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
        _beat_this.update(modelo=Audio2Frames(str(ckpt), device=dispositivo),
                          post=Postprocessor(type="minimal", fps=BEAT_THIS_FPS), dispositivo=dispositivo)
    return _beat_this


def detectar_pulsos(x, sr):
    """x: (canales, N). Devuelve pulsos (s), marca de downbeat y confianza (0–1) de cada pulso."""
    import torch
    bt = _cargar_beat_this()
    mono = np.asarray(x, dtype=np.float64).mean(axis=0)
    logit_pulso, logit_db = bt["modelo"](mono, sr)
    pulsos, downbeats = bt["post"](logit_pulso, logit_db)
    pulsos, downbeats = np.asarray(pulsos, float), np.asarray(downbeats, float)
    activ = torch.sigmoid(logit_pulso).cpu().numpy()
    cuadros = np.clip(np.round(pulsos * BEAT_THIS_FPS).astype(int), 0, len(activ) - 1)
    es_db = np.array([bool(len(downbeats)) and np.min(np.abs(downbeats - t)) < 0.5 / BEAT_THIS_FPS for t in pulsos])
    return {"pulsos": pulsos, "downbeat": es_db, "confianza": activ[cuadros] if len(pulsos) else np.empty(0),
            "fuente": f"detección automática (Beat This! final0, {bt['dispositivo']})"}


# ---------- Marcas corregidas a mano: <tema>.clic.csv ----------
def _segundos(texto):
    """'62.5', '1:02.5', '0:01:02.5' → segundos. Otros formatos de regla de Reaper → ValueError con ayuda."""
    t = texto.strip().replace(",", ".")
    if re.fullmatch(r"\d+(\.\d+)?", t):
        return float(t)
    partes = t.split(":")
    if len(partes) in (2, 3) and all(re.fullmatch(r"\d+(\.\d+)?", p) for p in partes):
        segs = 0.0
        for p in partes:
            segs = segs * 60 + float(p)
        return segs
    if re.fullmatch(r"\d+\.\d+\.\d+", t):
        raise ValueError(f"«{texto}» está en compases.tiempos: en Reaper pon la regla en Minutos:Segundos antes de exportar")
    if re.fullmatch(r"\d+:\d+:\d+:\d+", t):
        raise ValueError(f"«{texto}» está en horas:min:seg:cuadros: en Reaper pon la regla en Minutos:Segundos antes de exportar")
    raise ValueError(f"no entiendo el tiempo «{texto}»")


def _es_downbeat(nombre):
    """Convención: un marcador cuyo nombre empieza con C + número (C1, C12…) es el 1 del compás.
    También vale «12.1» (compás.tiempo con tiempo 1), «D» o «1»."""
    n = (nombre or "").strip().upper()
    return bool(re.fullmatch(r"C\s*\d+.*", n) or re.fullmatch(r"\d+\.1", n) or n in ("D", "1", "↓"))


def leer_clic_csv(ruta):
    """Acepta el CSV que exporta el gestor de regiones/marcadores de Reaper (columnas #, Name, Start…)
    o clic_marcas.csv de este notebook (columna segundos, downbeat). Devuelve (pulsos, downbeat, avisos)."""
    texto = Path(ruta).read_text(encoding="utf-8-sig", errors="replace")
    try:
        dialecto = csv.Sniffer().sniff(texto[:2000], delimiters=",;\t")
    except csv.Error:
        dialecto = csv.excel
    filas = [f for f in csv.reader(texto.splitlines(), dialecto) if any(c.strip() for c in f)]
    if not filas:
        raise ValueError("el archivo está vacío")
    cab = [c.strip().lower() for c in filas[0]]

    def col(*nombres):
        return next((i for i, c in enumerate(cab) if c in nombres), None)

    i_t = col("start", "position", "posición", "posicion", "inicio", "segundos", "seconds", "time", "tiempo")
    i_n = col("name", "nombre")
    i_id = col("#", "id")
    i_db = col("downbeat")
    if i_t is None:
        raise ValueError(f"no encuentro la columna de tiempo (Start o segundos) en la cabecera {filas[0]}")
    pulsos, flags, avisos = [], [], []
    for k, f in enumerate(filas[1:], start=2):
        if i_id is not None and len(f) > i_id and f[i_id].strip().upper().startswith("R"):
            continue  # regiones de Reaper: no son pulsos
        try:
            t = _segundos(f[i_t])
        except (ValueError, IndexError) as e:
            avisos.append(f"línea {k} ignorada: {e}")
            continue
        if i_db is not None and len(f) > i_db:
            db = f[i_db].strip().lower() in ("1", "si", "sí", "x", "true", "downbeat")
        else:
            db = _es_downbeat(f[i_n] if i_n is not None and len(f) > i_n else "")
        pulsos.append(t)
        flags.append(db)
    if len(pulsos) < 2:
        raise ValueError("hay menos de 2 marcas válidas")
    orden = np.argsort(pulsos)
    pulsos, flags = np.asarray(pulsos)[orden], np.asarray(flags)[orden]
    if not flags.any():
        avisos.append("ningún marcador está marcado como 1 del compás (nombre «C<número>»): "
                      "el clic usará el mismo tono en todos los pulsos")
    return pulsos, flags, avisos


def buscar_clic_csv(tema, carpeta_audio=None):
    """<tema>.clic.csv junto al audio (Colab: entrada/; Kaggle: cualquier Dataset de entrada)."""
    for base in [carpeta_audio, ENTRADA]:
        if base is not None and (Path(base) / f"{tema}.clic.csv").exists():
            return Path(base) / f"{tema}.clic.csv"
    if globals().get("ENTORNO") == "kaggle":  # Kaggle: en cualquier Dataset de entrada (subcarpetas incluidas)
        for base in (ENTRADA, Path("/kaggle/input")):
            if Path(base).is_dir():
                hallado = next(iter(sorted(Path(base).rglob(f"{tema}.clic.csv"))), None)
                if hallado is not None:
                    return hallado
    return None


# ---------- Compases, sonido y archivos ----------
def numerar_compases(flags):
    """Número de compás (0 = anacrusa antes del primer 1) y tiempo dentro del compás."""
    compas, tiempo, c, t = [], [], 0, 0
    for f in flags:
        if f:
            c, t = c + 1, 1
        else:
            t += 1
        compas.append(c)
        tiempo.append(t)
    return compas, tiempo


def sintetizar_clic(muestras, flags, n, sr):
    """Clic estéreo de N muestras: cada golpe EMPIEZA exactamente en su muestra (coseno, sin rampa)."""
    y = np.zeros(n, dtype=np.float64)
    largo = int(CLIC_DURACION_S * sr)
    t = np.arange(largo) / sr
    env = np.exp(-t / (CLIC_DURACION_S / 5))
    golpe_db = CLIC_NIVEL_DOWNBEAT * np.cos(2 * np.pi * CLIC_TONO_DOWNBEAT * t) * env
    golpe = CLIC_NIVEL_PULSO * np.cos(2 * np.pi * CLIC_TONO_PULSO * t) * env
    for m, f in zip(muestras, flags):
        if 0 <= m < n:
            g = golpe_db if f else golpe
            fin = min(n, m + largo)
            y[m:fin] += g[:fin - m]
    return np.stack([y, y]).astype(np.float32)


def escribir_marcas_csv(ruta, pulsos, muestras, flags, conf, sr):
    compas, tiempo = numerar_compases(flags)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["n", "segundos", f"muestra_{sr}", "downbeat", "compas", "tiempo", "confianza"])
        for i, (t, m, d, c, ti) in enumerate(zip(pulsos, muestras, flags, compas, tiempo), 1):
            w.writerow([i, f"{t:.6f}", int(m), int(bool(d)), c, ti, "" if conf is None else f"{conf[i - 1]:.2f}"])


def _mm_ss_ms(t):
    return f"{int(t // 60)}:{t % 60:06.3f}"


def escribir_marcadores_reaper(ruta, pulsos, flags):
    """Mismo esquema que exporta el gestor de marcadores de Reaper (#, Name, Start, End, Length),
    con Start en Minutos:Segundos. El 1 de cada compás se llama C<n>; los demás pulsos <n>.<tiempo>."""
    compas, tiempo = numerar_compases(flags)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["#", "Name", "Start", "End", "Length"])
        for i, (t, d, c, ti) in enumerate(zip(pulsos, flags, compas, tiempo), 1):
            w.writerow([f"M{i}", f"C{c}" if d else f"{c}.{ti}", _mm_ss_ms(t), "", ""])


def escribir_midi(ruta, pulsos, flags):
    """clic.mid: mapa de tempo (una negra = un pulso detectado), compás n/4 en cada cambio,
    marcador C<n> en cada 1 y una nota por pulso (canal 10: 76 en el 1, 77 en el resto)."""
    import mido
    pulsos = np.asarray(pulsos, float)
    flags = np.asarray(flags, bool)
    mid = mido.MidiFile(type=1, ticks_per_beat=MIDI_PPQ)
    pista_t, pista_n = mido.MidiTrack(), mido.MidiTrack()
    mid.tracks += [pista_t, pista_n]
    pista_t.append(mido.MetaMessage("track_name", name="Clic (mapa de tempo)", time=0))
    pista_n.append(mido.MetaMessage("track_name", name="Clic", time=0))
    # antes del primer pulso: k negras que llenan exactamente el silencio inicial
    k = 0 if pulsos[0] <= 0 else max(1, math.ceil(pulsos[0] / 2.0))
    idx_db = np.flatnonzero(flags)
    primero_db = int(idx_db[0]) if len(idx_db) else 0
    eventos = []  # (tick, orden, mensaje) en la pista de tempo
    if k:
        eventos.append((0, 0, mido.MetaMessage("set_tempo", tempo=int(round(pulsos[0] / k * 1e6)))))
    largo_previo = k + primero_db
    if largo_previo:
        eventos.append((0, 1, mido.MetaMessage("time_signature", numerator=min(255, largo_previo), denominator=4)))
    ultimo_compas = None
    compas, _ = numerar_compases(flags)
    for i, t in enumerate(pulsos):
        tick = (k + i) * MIDI_PPQ
        ibi = pulsos[i + 1] - t if i + 1 < len(pulsos) else (pulsos[i] - pulsos[i - 1] if i else 0.5)
        eventos.append((tick, 0, mido.MetaMessage("set_tempo", tempo=int(round(min(ibi, 16.0) * 1e6)))))
        if flags[i]:
            sig = (idx_db[idx_db > i][0] - i) if (idx_db > i).any() else (ultimo_compas or 4)
            if sig != ultimo_compas:
                eventos.append((tick, 1, mido.MetaMessage("time_signature", numerator=int(min(255, sig)), denominator=4)))
                ultimo_compas = sig
            eventos.append((tick, 2, mido.MetaMessage("marker", text=f"C{compas[i]}")))
    previo = 0
    for tick, _, msg in sorted(eventos, key=lambda e: (e[0], e[1])):
        pista_t.append(msg.copy(time=tick - previo))
        previo = tick
    previo, dur = 0, MIDI_PPQ // 8
    for i in range(len(pulsos)):
        tick = (k + i) * MIDI_PPQ
        nota, vel = (76, 120) if flags[i] else (77, 90)
        pista_n.append(mido.Message("note_on", channel=9, note=nota, velocity=vel, time=tick - previo))
        pista_n.append(mido.Message("note_off", channel=9, note=nota, velocity=0, time=dur))
        previo = tick + dur
    mid.save(str(ruta))


# ---------- Zonas dudosas ----------
def _zonas(puntos, unir_s=2.0, margen=0.5):
    zonas = []
    for a, b in sorted(puntos):
        a, b = max(0.0, a - margen), b + margen
        if zonas and a - zonas[-1][1] < unir_s:
            zonas[-1][1] = max(zonas[-1][1], b)
        else:
            zonas.append([a, b])
    return zonas


def zonas_dudosas(pulsos, conf, duracion, bateria=None, sr=SR):
    """Lista de (tipo, inicio_s, fin_s) para clic_informe.txt."""
    res = []
    if len(pulsos) < 4:
        return [("muy pocos pulsos detectados", 0.0, duracion)]
    ibi = np.diff(pulsos)
    med = float(np.median(ibi))
    hueco = ibi > DUDA_HUECO * med
    huecos = [(pulsos[i], pulsos[i + 1]) for i in np.flatnonzero(hueco)]
    res += [("hueco sin pulsos (¿parada, calderón, silencio?)", a, b) for a, b in _zonas(huecos, 0.1, 0.0)]
    # cambio brusco: un pulso aislado que se aparta de sus vecinos, o un escalón de tempo (4 antes vs 4 después)
    brusco = []
    for i in range(len(ibi)):
        if hueco[max(0, i - 4):i + 5].any():
            continue
        vecinos = np.median(ibi[max(0, i - 4):i + 5])
        antes, despues = ibi[max(0, i - 4):i], ibi[i:i + 4]
        escalon = len(antes) == 4 and len(despues) == 4 and abs(np.median(despues) / np.median(antes) - 1) > DUDA_CAMBIO_BRUSCO
        if abs(ibi[i] / vecinos - 1) > DUDA_CAMBIO_BRUSCO or escalon:
            brusco.append((pulsos[i], pulsos[i + 1]))
    res += [("cambio brusco de tempo", a, b) for a, b in _zonas(brusco)]
    # rubato: el tempo varía de pulso a pulso (promedio de |Δintervalo| relativo en 8 pulsos), sin huecos
    rub = []
    for i in range(len(ibi) - 8):
        v = ibi[i:i + 9]
        if hueco[i:i + 9].any():
            continue
        if np.mean(np.abs(np.diff(v)) / v[:-1]) > DUDA_RUBATO_CV:
            rub.append((pulsos[i], pulsos[i + 8]))
    res += [("rubato / tempo inestable", a, b) for a, b in _zonas(rub)]
    if conf is not None and len(conf):
        baja = [(pulsos[i], pulsos[i]) for i in np.flatnonzero(conf < DUDA_CONFIANZA)]
        res += [("baja confianza de la detección", a, b) for a, b in _zonas(baja)]
    if pulsos[0] > 2.0:
        res.append(("sin pulsos al inicio", 0.0, float(pulsos[0])))
    if duracion - pulsos[-1] > 2.0:
        res.append(("sin pulsos al final", float(pulsos[-1]), duracion))
    if bateria is not None:
        t, rms = _rms_ventanas(bateria, sr)
        ref = np.percentile(rms[rms > 0], 75) if (rms > 0).any() else 0
        bajo = rms < ref * 10 ** (SIN_BATERIA_DB / 20)
        tramos, ini = [], None
        for ti, b in zip(t, bajo):
            if b and ini is None:
                ini = ti
            if not b and ini is not None:
                tramos.append((ini, ti)); ini = None
        if ini is not None:
            tramos.append((ini, duracion))
        res += [("parte sin batería (el clic se apoya en otros instrumentos)", a, b)
                for a, b in tramos if b - a >= 4.0]
    return sorted(res, key=lambda z: z[1])


# ---------- Generación completa ----------
def generar_clic(tema, mezcla, sr_mezcla, n_salida, carpeta, carpeta_audio=None, bateria=None,
                 sr_salida=None, mp3=None):
    """Detecta (o lee <tema>.clic.csv), escribe carpeta/clic/ y devuelve un dict con el resumen.
    n_salida y sr_salida: largo y frecuencia de los stems (el clic dura exactamente lo mismo)."""
    sr_salida = sr_salida or SR_SALIDA
    destino = Path(carpeta) / "clic"
    destino.mkdir(parents=True, exist_ok=True)
    manual = buscar_clic_csv(tema, carpeta_audio)
    avisos = []
    if manual is not None:
        pulsos, flags, avisos = leer_clic_csv(manual)
        conf, fuente = None, f"marcas corregidas a mano ({manual.name})"
    else:
        d = detectar_pulsos(mezcla, sr_mezcla)
        pulsos, flags, conf, fuente = d["pulsos"], d["downbeat"], d["confianza"], d["fuente"]
    duracion = n_salida / sr_salida
    dentro = pulsos < duracion
    pulsos, flags = pulsos[dentro], flags[dentro]
    conf = conf[dentro] if conf is not None else None
    muestras = np.round(pulsos * sr_salida).astype(np.int64)
    ruta_audio = escribir_audio(destino / "clic", sintetizar_clic(muestras, flags, n_salida, sr_salida), sr_salida, mp3)
    # verificación: mismo largo que los stems y cada golpe en su muestra exacta (desfase 0)
    desfase = None
    if ruta_audio.suffix == ".flac" and len(muestras):
        y = leer_escrito(ruta_audio)[0]
        largo_ok = y.shape[0] == n_salida
        onsets = [int(np.flatnonzero(np.abs(y[m:m + 64]) > 1e-4)[0]) if (np.abs(y[m:m + 64]) > 1e-4).any() else 99
                  for m in muestras[:50]]
        desfase = max(onsets) if largo_ok else None
    escribir_marcas_csv(destino / "clic_marcas.csv", pulsos, muestras, flags, conf, sr_salida)
    escribir_marcadores_reaper(destino / "clic_marcadores_reaper.csv", pulsos, flags)
    escribir_midi(destino / "clic.mid", pulsos, flags)
    zonas = zonas_dudosas(pulsos, conf, duracion, bateria, sr_mezcla if bateria is not None else SR)
    info = {"fuente": fuente, "pulsos": int(len(pulsos)), "compases": int(np.sum(flags)),
            "bpm_mediana": round(60 / float(np.median(np.diff(pulsos))), 1) if len(pulsos) > 1 else None,
            "desfase_muestras": desfase, "zonas_dudosas": len(zonas), "avisos": avisos, "sr": sr_salida,
            "muestras_totales": int(n_salida)}
    (destino / "clic_informe.txt").write_text(texto_informe_clic(tema, info, pulsos, flags, zonas, bateria is not None),
                                              encoding="utf-8")
    return info


def texto_informe_clic(tema, info, pulsos, flags, zonas, hubo_bateria):
    compas, tiempo = numerar_compases(flags)
    largos = {}
    for c in set(compas) - {0}:
        n = sum(1 for x in compas if x == c)
        largos[n] = largos.get(n, 0) + 1
    ibi = np.diff(pulsos) if len(pulsos) > 1 else np.array([0.5])
    L = [f"CLIC — {tema}", "",
         f"Fuente de los pulsos: {info['fuente']}",
         f"Pulsos: {info['pulsos']} · compases: {info['compases']} · tempo mediano: {info['bpm_mediana']} BPM "
         f"(rango {60 / ibi.max():.0f}–{60 / ibi.min():.0f})",
         "Compases según cantidad de tiempos: " + (", ".join(f"{n} tiempos: {c}" for n, c in sorted(largos.items())) or "—"),
         f"Alineación: el clic empieza en la muestra 0 y dura lo mismo que los stems "
         f"({info['muestras_totales']} muestras a {info['sr']} Hz); "
         + ("cada golpe verificado en su muestra exacta (desfase 0 muestras)." if info["desfase_muestras"] == 0
            else "⚠ no se pudo verificar el desfase exacto (MP3 agrega retardo de codificación; usa FLAC para el DAW)."),
         "Precisión de la detección automática: cuadros de 20 ms (±10 ms). Las marcas corregidas a mano se usan tal cual.",
         "Sonidos: tono agudo (Sol6) en el 1 de cada compás; tono medio (Do6) en los demás pulsos."]
    for a in info.get("avisos", []):
        L.append(f"⚠ {a}")
    L += ["", f"ZONAS DUDOSAS ({len(zonas)}) — escúchalas y corrígelas en Reaper si hace falta:"]
    if not hubo_bateria:
        L.append("  (sin stem de batería en este modo: no se buscan partes sin batería)")
    if not zonas:
        L.append("  ninguna")
    for tipo, a, b in zonas:
        L.append(f"  {mm_ss(a):>7s} – {mm_ss(b):>7s}  {tipo}")
    L += ["", "ARCHIVOS",
          "  clic.flac                    el clic (mismo largo, frecuencia y formato que los stems)",
          "  clic_marcas.csv              cada pulso: segundos, muestra, downbeat, compás, tiempo, confianza",
          "  clic.mid                     mapa de tempo + compases + una nota por pulso (para Reaper)",
          "  clic_marcadores_reaper.csv   un marcador por pulso, con el formato del gestor de marcadores de Reaper",
          "",
          "REAPER 7 — USAR EL MAPA DE TEMPO (clic.mid)",
          "  1. Proyecto nuevo. Arrastra los stems (o Insertar › Archivo multimedia) al inicio (0:00).",
          "  2. Insertar › Archivo multimedia… (Insert › Media file…) › elige clic/clic.mid.",
          "     Cuando Reaper pregunte, marca «Import MIDI file tempo map» (importar el mapa de tempo del MIDI).",
          "  3. Pon el ítem MIDI en 0:00. La regla de compases y la grilla quedan alineadas con los pulsos.",
          "  4. Para que el audio NO se estire con el tempo: selecciona los ítems de audio › Propiedades del ítem",
          "     (F2) › Timebase: Time. Así el mapa de tempo solo mueve la grilla.",
          "  (Cómo pregunta Reaper por el mapa de tempo lo verifiqué en foros, no en el manual: si no aparece el",
          "   diálogo, revisa Preferencias › Media › MIDI, opción de importar tempo de archivos MIDI.)",
          "",
          "REAPER 7 — CORREGIR LAS MARCAS A MANO",
          "  A. Importar los marcadores: Ver › Gestor de regiones/marcadores (View › Region/Marker Manager) ›",
          "     clic derecho › Import… › clic_marcadores_reaper.csv. ⚠ No pude verificar que Reaper 7 importe",
          "     este CSV tal cual; si no lo acepta, crea los marcadores con la grilla del paso anterior.",
          "  B. Mueve los marcadores que estén corridos. Convención: el 1 de cada compás se llama C<número>",
          "     (C1, C2…); los demás pulsos, como quieras (por ejemplo 12.3). Agrega o borra marcadores libremente.",
          "  C. Pon la regla en Minutos:Segundos (clic derecho en la regla) y exporta: Gestor de regiones/marcadores ›",
          "     clic derecho › Export… › guarda como «<tema>.clic.csv».",
          "  D. Sube ese archivo junto al audio del tema (Colab: Drive/stems/entrada/; Kaggle: en el Dataset) y corre",
          "     con ACCION = «solo clic»: el notebook usa tus marcas en vez de la detección.",
          "  El lector acepta tiempos en segundos, m:ss.mmm o h:mm:ss.mmm, separados por coma, punto y coma o tab.",
          "  No acepta la regla en compases.tiempos ni en cuadros: avisa en este informe qué línea ignoró.",
          "",
          "ABLETON LIVE 10",
          "  Live 10 no importa mapas de tempo desde MIDI: usa clic.flac en una pista de audio, con Warp",
          "  desactivado, junto a los stems (todo desde 1.1.1 / 0:00).",
          "",
          "Nada de esto vale hasta escucharlo contra la canción."]
    return "\n".join(L) + "\n"

print("✓ Clic listo")


## 9 · Procesar
Todos los temas de la carpeta de entrada. El avance se ve en las barras, en una línea fija con la etapa actual y en `salida/progreso.txt` (para mirarlo desde otro dispositivo); en Kaggle además queda una línea por etapa en el log y, al final, un `.zip` por tema en `stems/zips/`. Si un tema falla, se registra en `salida/errores.txt` y se sigue con el siguiente.

In [ ]:
#@title 9 · Procesar todos los temas de la carpeta de entrada
# Avance: barras (temas / etapas / segmentos), una línea fija con la etapa actual y
# Drive/stems/salida/progreso.txt, que se reescribe en cada etapa (y queda con el resumen).
ENTORNO = globals().get("ENTORNO", "colab")
SALIDA_INFO = globals().get("SALIDA_INFO") or SALIDA  # progreso.txt y errores.txt (Kaggle: en el Output)
ZIPS = globals().get("ZIPS")                         # Kaggle: un zip por tema en el Output; Colab: None
MOSTRAR_AVANCE = not globals().get("KAGGLE_SEGUNDO_PLANO", False)  # en segundo plano no hay widgets


def destino_hz():
    if DESTINO_HZ is None:
        return None
    if not 380.0 <= DESTINO_HZ <= 500.0:
        raise Detenido(f"La referencia {DESTINO_HZ} Hz no parece válida (usa algo entre 380 y 500 Hz).")
    return DESTINO_HZ


# --- Decisión de afinación (escala de incertidumbre de centrail §3: ≤1,5 alta / 1,5–6 media / >6 alta) ---
UNC_MAXIMA_AUTO = 6.0  # sobre esto la medición no es confiable: no se corrige automáticamente
R_MINIMO = 0.05        # R bajo solo pesa si además la incertidumbre supera el umbral (R es informativo)


def _sin_tildes(texto):
    import unicodedata
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def leer_afinacion_manual(tema, carpeta=None):
    """Lee entrada/<tema>.afinacion.txt. Devuelve (orden, avisos). Órdenes aceptadas (una por archivo):
       corregir <cents>   ·   detectado <Hz>   ·   no_afinar"""
    import re
    archivo = (carpeta or ENTRADA) / f"{tema}.afinacion.txt"
    if not archivo.exists():
        return None, []
    orden, avisos = None, []
    ayuda = "usa: «corregir -8.0», «detectado 442.1» o «no_afinar»"
    for n, linea in enumerate(archivo.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
        t = _sin_tildes(linea.split("#")[0]).strip().lower().replace(",", ".")
        t = re.sub(r"[:=]", " ", t).strip()
        if not t:
            continue
        nueva = None
        m = re.fullmatch(r"(?:corregir|correccion|corrige)\s+([+-]?\d+(?:\.\d+)?)\s*(?:c|¢|cents?)?", t)
        if m:
            cents = float(m.group(1))
            if abs(cents) > 100:
                avisos.append(f"línea {n} ignorada: {cents:+g} ¢ es más de un semitono ({ayuda})")
                continue
            nueva = {"tipo": "corregir", "cents": cents}
        m = m or re.fullmatch(r"(?:detectado|detectada|referencia)\s+(\d+(?:\.\d+)?)\s*(?:hz)?", t)
        if m and nueva is None:
            hz = float(m.group(1))
            if not 380 <= hz <= 500:
                avisos.append(f"línea {n} ignorada: {hz:g} Hz fuera de 380–500 Hz ({ayuda})")
                continue
            nueva = {"tipo": "detectado", "hz": hz}
        if nueva is None and re.fullmatch(r"no[ _-]?afinar", t):
            nueva = {"tipo": "no_afinar"}
        if nueva is None:
            avisos.append(f"línea {n} ignorada: «{linea.strip()}» ({ayuda})")
        elif orden is not None:
            avisos.append(f"línea {n} ignorada: ya había una orden antes («{orden['linea']}»)")
        else:
            orden = {**nueva, "linea": linea.strip()}
    return orden, avisos


def clasificar_medicion(m):
    if m["unc"] > UNC_MAXIMA_AUTO:
        return "no confiable"
    if m["R"] < R_MINIMO and m["unc"] > UMBRAL_CENTS:
        return "no confiable"
    return "media" if m["unc"] > UMBRAL_CENTS else "alta"


def medir_sin_bateria(x, nombre, tiempos):
    with Etapa(nombre, tiempos):
        sep = cargar_modelo(MODELO_DEMUCS)
        try:
            bateria = salidas_modelo(sep, x)["drums"]
        finally:
            liberar_modelo(sep)
        return medir_afinacion(x - bateria)  # el audio sin batería solo sirve para medir


def _aplicar_correccion(mezcla, shift, info, tiempos):
    """Rubber Band UNA vez sobre la mezcla ORIGINAL (nunca sobre una ya corregida) + seguridad de nivel."""
    with Etapa(f"afinar {shift:+.2f} ¢ (Rubber Band R3)", tiempos):
        afinada = afinar(mezcla, shift)
    techo = min(1.0, info["pico_original"])
    p = pico(afinada)
    ganancia = techo / p if p > techo else 1.0
    if ganancia < 1.0:
        afinada = (afinada * ganancia).astype(np.float32)
    info.update(corregido=True, shift=shift, ganancia_afinar_db=dbfs(ganancia))
    return afinada


def preparar_afinacion(mezcla, info, tiempos, tema, carpeta=None):
    """Decide y aplica la corrección ANTES de separar. Devuelve la mezcla con la que trabaja todo lo demás.
    Prioridad: archivo manual > medición. Medición incierta → lazo cerrado (una iteración, desde el original)."""
    dest = destino_hz()
    info.update(destino_hz=dest, corregido=False, shift=0.0, motor=MOTOR_RB, pico_original=pico(mezcla),
                fuente="—", razon="", avisos_afinacion=[])
    manual, avisos = leer_afinacion_manual(tema, carpeta)
    info["avisos_manual"] = avisos
    for a in avisos:
        print("    ⚠", f"{tema}.afinacion.txt:", a)
    if manual:
        info["manual"] = manual
        if manual["tipo"] == "no_afinar":
            info.update(fuente="manual", razon=f"«{manual['linea']}» en {tema}.afinacion.txt: se separa sin corregir")
            return mezcla
        if manual["tipo"] == "detectado":
            if dest is None:
                info.update(fuente="manual", razon="«detectado» necesita una referencia destino, pero AFINAR está "
                                                   "desactivado: no se corrigió")
                info["avisos_afinacion"].append("⚠ orden manual sin destino: no se aplicó")
                return mezcla
            shift = 1200 * math.log2(dest / manual["hz"])
        else:
            shift = manual["cents"]
        info.update(fuente="manual", razon=f"«{manual['linea']}» en {tema}.afinacion.txt (tiene prioridad sobre la medición)")
        print(f"    corrección manual: {shift:+.2f} ¢")
        return _aplicar_correccion(mezcla, shift, info, tiempos) if abs(shift) > 0.01 else mezcla
    if dest is None:
        return mezcla

    m1 = info["medicion"] = medir_sin_bateria(mezcla, "medir afinación 1 (mezcla − batería)", tiempos)
    shift1 = 1200 * math.log2(dest / m1["ref_hz"])
    nivel = info["nivel_medicion"] = clasificar_medicion(m1)
    print(f"    medición 1: {m1['ref_hz']:.2f} Hz (±{m1['unc']:.2f} ¢, R {100 * m1['R']:.1f} %, "
          f"deriva {m1['deriva_max']:.1f} ¢) → {shift1:+.2f} ¢ · confianza {nivel}")
    info["shift_medicion1"] = shift1
    if nivel == "no confiable":
        info.update(fuente="ninguna", razon=f"medición no confiable (±{m1['unc']:.2f} ¢, R {100 * m1['R']:.1f} %): "
                    f"no se corrige automáticamente; si hace falta, corrige a mano con {tema}.afinacion.txt")
        info["avisos_afinacion"].append("⚠ medición no confiable: no se corrigió")
        return mezcla
    if abs(shift1) <= UMBRAL_CENTS:
        info.update(fuente="medición 1", razon=f"ya estaba afinado (|{shift1:+.2f}| ≤ {UMBRAL_CENTS:g} ¢), no se tocó")
        if nivel == "media":
            info["avisos_afinacion"].append(f"⚠ medición incierta (±{m1['unc']:.2f} ¢ > umbral)")
        return mezcla
    y1 = _aplicar_correccion(mezcla, shift1, info, tiempos)
    if nivel == "alta":
        info.update(fuente="medición 1", razon=f"medición confiable (±{m1['unc']:.2f} ¢ ≤ {UMBRAL_CENTS:g} ¢)")
        return y1

    # Incertidumbre media → lazo cerrado: medir la mezcla ya corregida (máximo una iteración)
    m2 = info["medicion2"] = medir_sin_bateria(y1, "medir afinación 2 (lazo cerrado)", tiempos)
    r = info["residuo2"] = 1200 * math.log2(m2["ref_hz"] / dest)
    print(f"    medición 2: residuo {r:+.2f} ¢ respecto del destino (±{m2['unc']:.2f} ¢)")
    if abs(r) <= UMBRAL_CENTS:
        info.update(fuente="medición 1", razon=f"medición 1 incierta (±{m1['unc']:.2f} ¢), confirmada por la "
                    f"medición 2 (residuo {r:+.2f} ¢ dentro de ±{UMBRAL_CENTS:g} ¢)")
        return y1
    est2 = shift1 - r  # corrección que indica la medición 2, referida a la mezcla original
    if m2["unc"] <= UMBRAL_CENTS:
        final, fuente = est2, "medición 2"
        razon = (f"la medición 1 era incierta (±{m1['unc']:.2f} ¢) y la medición 2 es confiable (±{m2['unc']:.2f} ¢): "
                 f"se recalculó la corrección con la medición 2 ({shift1:+.2f} − ({r:+.2f}) = {est2:+.2f} ¢)")
    else:
        w1, w2 = 1 / max(m1["unc"], 0.1) ** 2, 1 / max(m2["unc"], 0.1) ** 2
        final, fuente = (w1 * shift1 + w2 * est2) / (w1 + w2), "combinación"
        razon = (f"ambas mediciones inciertas (±{m1['unc']:.2f} y ±{m2['unc']:.2f} ¢): promedio ponderado por "
                 f"1/incertidumbre² de {shift1:+.2f} (medición 1) y {est2:+.2f} ¢ (medición 2)")
        info["avisos_afinacion"].append("⚠ corrección por combinación de dos mediciones inciertas")
    del y1
    info.update(fuente=fuente, razon=razon + "; aplicada desde la mezcla ORIGINAL (sin encadenar Rubber Band)")
    print(f"    corrección final ({fuente}): {final:+.2f} ¢")
    return _aplicar_correccion(mezcla, final, info, tiempos)


def escribir_y_verificar(salidas, referencia, nombres_suma, carpeta, info, tiempos):
    """Remuestrea (si toca), evita recorte con UNA ganancia común, escribe y hace la prueba de cancelación."""
    with Etapa(f"escribir {len(salidas)} archivos ({FORMATO}, {SR_SALIDA} Hz)", tiempos):
        ref = a_frecuencia_salida(referencia)
        listos = {k: a_frecuencia_salida(v) for k, v in salidas.items()}
        p = max(pico(v) for v in listos.values())
        techo = 1.0 - 2.0 ** -23
        g = techo / p if p > techo else 1.0
        info["ganancia_recorte_db"] = dbfs(g)
        rutas = {}
        for j, (k, v) in enumerate(listos.items()):
            avisar_segmento(j, len(listos))
            rutas[k] = escribir_audio(carpeta / k, v * g if g < 1 else v, SR_SALIDA)
        avisar_segmento(len(listos), len(listos))
        ref = ref.astype(np.float64) * g
        del listos
    with Etapa("prueba de cancelación", tiempos):
        suma = None
        for j, k in enumerate(nombres_suma):
            avisar_segmento(j + 1, len(nombres_suma))
            y = leer_escrito(rutas[k])
            if suma is None:
                suma = y
            else:
                n = min(suma.shape[1], y.shape[1])
                suma = suma[:, :n] + y[:, :n]
        n = min(suma.shape[1], ref.shape[1])
        residuo = dbfs(pico(suma[:, :n] - ref[:, :n]))
    info["residuo_dbfs"] = residuo
    info["n_salida"] = int(ref.shape[1])  # largo de los stems escritos (el clic dura exactamente esto)
    if ES_MP3:
        info["veredicto_cancelacion"] = "MP3: la suma no es exacta por diseño (solo para escuchar)"
    elif residuo < RESIDUO_MAX_DBFS:
        info["veredicto_cancelacion"] = f"OK (bajo {RESIDUO_MAX_DBFS:g} dBFS)"
    else:
        info["veredicto_cancelacion"] = f"⚠ ADVERTENCIA: sobre {RESIDUO_MAX_DBFS:g} dBFS, la suma no cuadra"
    print(f"    residuo de cancelación: {residuo:.1f} dBFS · {info['veredicto_cancelacion']}")
    return rutas


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in carpeta.iterdir() if f.is_file()) / 1e6


def mover_a_anteriores(destino, tema):
    """MUEVE (nunca borra) la carpeta de salida de un tema a salida/_anteriores/<tema>_<fecha>/."""
    anteriores = SALIDA / "_anteriores"
    anteriores.mkdir(exist_ok=True)
    sello = time.strftime("%Y-%m-%d_%H%M")
    nuevo, k = anteriores / f"{tema}_{sello}", 2
    while nuevo.exists():
        nuevo, k = anteriores / f"{tema}_{sello}_{k}", k + 1
    os.rename(destino, nuevo)  # mismo Drive: renombrar; si fallara, se detiene sin copiar ni borrar nada
    return nuevo


def _resumen_clic(c):
    if not c:
        return None
    if "error" in c:
        return "ERROR"
    return {"pulsos": c["pulsos"], "bpm": c["bpm_mediana"], "zonas": c["zonas_dudosas"],
            "manual": c["fuente"].startswith("marcas"), "desfase": c["desfase_muestras"]}


def comprimir_tema(carpeta_tema, carpeta_zips, nombre_zip=None, prefijo=None):
    """Kaggle: zip de una carpeta (con subcarpetas, sin comprimir: FLAC ya lo está), escrito como
    .parcial y renombrado. prefijo = ruta dentro del zip (por defecto, el nombre de la carpeta)."""
    import zipfile
    prefijo = prefijo or carpeta_tema.name
    destino = carpeta_zips / (nombre_zip or f"{carpeta_tema.name}.zip")
    parcial = destino.with_name(destino.name + ".parcial")
    with zipfile.ZipFile(parcial, "w", compression=zipfile.ZIP_STORED) as z:
        for f in sorted(carpeta_tema.rglob("*")):
            if f.is_file():
                z.write(f, f"{prefijo}/{f.relative_to(carpeta_tema).as_posix()}")
    os.replace(parcial, destino)
    return destino


def verificar_zip(zip_ruta, carpeta_tema, prefijo=None):
    """True si el zip se lee entero (CRC) y trae todos los archivos de la carpeta con el mismo tamaño."""
    import zipfile
    prefijo = prefijo or carpeta_tema.name
    try:
        with zipfile.ZipFile(zip_ruta) as z:
            if z.testzip() is not None:
                return False
            en_zip = {i.filename: i.file_size for i in z.infolist()}
    except (zipfile.BadZipFile, OSError):
        return False
    esperados = {f"{prefijo}/{f.relative_to(carpeta_tema).as_posix()}": f.stat().st_size
                 for f in carpeta_tema.rglob("*") if f.is_file()}
    return en_zip == esperados


def resumen_desde_zip(zip_ruta, tema):
    import zipfile
    with zipfile.ZipFile(zip_ruta) as z:
        return json.loads(z.read(f"{tema}/.proceso.json").decode("utf-8"))["resumen"]


def mover_zip_a_anteriores(zip_ruta, tema):
    """Kaggle + REPROCESAR: MUEVE (nunca borra) el zip anterior a zips/_anteriores/<tema>_<fecha>.zip."""
    anteriores = zip_ruta.parent / "_anteriores"
    anteriores.mkdir(exist_ok=True)
    sello = time.strftime("%Y-%m-%d_%H%M")
    nuevo, k = anteriores / f"{tema}_{sello}.zip", 2
    while nuevo.exists():
        nuevo, k = anteriores / f"{tema}_{sello}_{k}.zip", k + 1
    os.rename(zip_ruta, nuevo)
    return nuevo


def procesar_tema(ruta):
    tema = ruta.stem
    destino = SALIDA / tema
    manifiesto = destino / ".proceso.json"
    zip_tema = ZIPS / f"{tema}.zip" if ZIPS is not None else None
    if zip_tema is not None and zip_tema.exists():  # Kaggle: el zip en el Output marca el tema como hecho
        if en_reprocesar(tema):
            movido = mover_zip_a_anteriores(zip_tema, tema)
            print(f"  REPROCESAR: el zip anterior se movió a {movido.relative_to(ZIPS)}")
        else:
            print("  ya estaba completo (su zip existe): se salta. Para rehacerlo, agrégalo a REPROCESAR.")
            return {**resumen_desde_zip(zip_tema, tema), "estado": "ya estaba"}
    if destino.exists() and en_reprocesar(tema):
        movida = mover_a_anteriores(destino, tema)
        print(f"  REPROCESAR: la salida anterior se movió a {movida.relative_to(SALIDA)}/")
    elif manifiesto.exists():
        datos = json.loads(manifiesto.read_text(encoding="utf-8"))
        print("  ya estaba completo: se salta. Para rehacerlo, agrégalo a REPROCESAR en la configuración "
              "(la carpeta actual se moverá a salida/_anteriores/, no se borra nada).")
        return {**datos["resumen"], "estado": "ya estaba"}

    t0 = time.time()
    tiempos, registro, segmentos = {}, {}, {}
    trabajo = TRABAJO / tema
    if trabajo.exists():
        borrar_seguro(trabajo)
    trabajo.mkdir(parents=True)
    try:
        with Etapa("decodificar", tiempos):
            mezcla, sr_nativa = leer_audio(ruta)
        info = {"archivo": ruta.name, "sr_nativa": sr_nativa, "modelos": registro, "tiempos": tiempos}
        mezcla_trabajo = preparar_afinacion(mezcla, info, tiempos, tema, ruta.parent)
        del mezcla

        stems, partes, guitarras = cascada(mezcla_trabajo, separador_real(registro, tiempos, segmentos),
                                           MODO, ORDEN_GUITARRAS, SEPARAR_ACUSTICA_ELECTRICA)
        if info["corregido"] and MODO == "completo":
            with Etapa("verificar afinación tras separar", tiempos):
                v = info["verificacion"] = medir_afinacion(mezcla_trabajo - stems["03 Bateria"])
            info["verif_cents"] = 1200 * math.log2(v["ref_hz"] / info["destino_hz"])
            info["verif_fuera"] = abs(info["verif_cents"]) > UMBRAL_CENTS
            if info["verif_fuera"]:
                print(f"    ⚠ verificación fuera del umbral: {info['verif_cents']:+.2f} ¢ (±{UMBRAL_CENTS:g} ¢)")

        salidas = ({"00 Mezcla afinada": mezcla_trabajo} if info["corregido"] else {}) | stems
        escribir_y_verificar(salidas, mezcla_trabajo, list(stems), trabajo, info, tiempos)
        # clic sobre la MISMA mezcla afinada; fuera de los stems sumables (carpeta clic/, no entra en la cancelación)
        with Etapa("clic (Beat This!)", tiempos):
            try:
                info["clic"] = generar_clic(tema, mezcla_trabajo, SR, info["n_salida"], trabajo, ruta.parent,
                                            stems.get("03 Bateria"))
                print(f"    clic: {info['clic']['pulsos']} pulsos · {info['clic']['bpm_mediana']} BPM · "
                      f"{info['clic']['zonas_dudosas']} zona(s) dudosa(s) · {info['clic']['fuente']}")
            except Exception as e:  # los stems ya están: no se pierden; el clic se rehace con «solo clic»
                info["clic"] = {"error": f"{type(e).__name__}: {e}"}
                print(f"    ⚠ el clic falló ({e}); los stems se guardan igual. Rehazlo luego con ACCION = «solo clic».")
        if partes is not None:
            (trabajo / "guitarras.txt").write_text(
                texto_guitarras(tema, partes, guitarras, ORDEN_GUITARRAS), encoding="utf-8")
        del stems, partes, guitarras, salidas
        (trabajo / "afinacion.txt").write_text(texto_afinacion(tema, info), encoding="utf-8")

        with Etapa("copiar a la carpeta de salida", tiempos):
            destino.mkdir(parents=True, exist_ok=True)
            for f in sorted(trabajo.iterdir()):
                if f.is_file():
                    shutil.copy2(f, destino / f.name)
                elif f.is_dir():  # clic/
                    shutil.copytree(f, destino / f.name, dirs_exist_ok=True)
        m = info.get("medicion")
        resumen = {"tema": tema, "estado": "listo",
                   "hz": round(m["ref_hz"], 2) if m else None,
                   "unc_cents": round(m["unc"], 2) if m else None,
                   "correccion_cents": round(info["shift"], 2) if info["corregido"] else 0.0,
                   "fuente": info["fuente"],
                   "verif_cents": round(info["verif_cents"], 2) if "verif_cents" in info else None,
                   "aviso_afinacion": bool(info.get("verif_fuera") or info["avisos_afinacion"]),
                   "residuo_dbfs": round(info["residuo_dbfs"], 1),
                   "clic": _resumen_clic(info.get("clic")),
                   "mb": round(tamano_mb(destino), 1), "minutos": round((time.time() - t0) / 60, 1)}
        manifiesto.write_text(json.dumps({  # se escribe al ÚLTIMO: marca el tema como completo
            "resumen": resumen, "modo": MODO, "orden_guitarras": ORDEN_GUITARRAS,
            "acustica_electrica": SEPARAR_ACUSTICA_ELECTRICA, "formato": FORMATO, "frecuencia": SR_SALIDA,
            "afinar_hz": DESTINO_HZ, "afinacion": {
                k: info.get(k) for k in ("fuente", "razon", "shift", "corregido", "nivel_medicion", "shift_medicion1",
                                         "residuo2", "verif_cents", "manual", "avisos_afinacion", "avisos_manual")},
            "clic": info.get("clic"), "modelos": registro, "segmentos_roformer": segmentos, "fecha": time.strftime("%Y-%m-%d %H:%M")},
            ensure_ascii=False, indent=2), encoding="utf-8")
        if zip_tema is not None:
            with Etapa("comprimir y verificar zip", tiempos):
                comprimir_tema(destino, ZIPS)
                if not verificar_zip(zip_tema, destino):
                    raise RuntimeError(f"El zip de {tema} no se verificó: se deja la carpeta en {destino}")
                resumen["zip_mb"] = round(zip_tema.stat().st_size / 1e6, 1)
                borrar_seguro(destino, tema_en_curso=tema)  # stems sueltos de /kaggle/temp: ya están en el zip
        return resumen
    finally:
        borrar_seguro(trabajo)


def etapas_previstas():
    """Cuántas etapas tendrá un tema con esta configuración (para la barra de etapas)."""
    n = 1 + 2 + 3 + 1 + (1 if ZIPS is not None else 0)  # decodificar · voz, karaoke · escribir, cancelación, copiar · clic · [zip]
    if DESTINO_HZ is not None:
        n += 2  # medir y (si hace falta) afinar
    if MODO == "completo":
        n += 3 + (1 if SEPARAR_ACUSTICA_ELECTRICA else 0)  # guitarras, punteo, [acústica], demucs
        n += 1 if DESTINO_HZ is not None else 0  # verificar afinación tras separar
    return n


def _clic_corto(c):
    if not c:
        return "—"
    if c == "ERROR":
        return "ERROR"
    return "manual" if c.get("manual") else f"{c['bpm']}"


FUENTE_CORTA = {"medición 1": "med1", "medición 2": "med2", "combinación": "comb", "manual": "manual",
                "ninguna": "ninguna", "—": "—"}


def texto_resumen(filas):
    L = [f"{'tema':28s} {'estado':10s} {'Hz':>7s} {'±¢':>5s} {'corr. ¢':>8s} {'fuente':>7s} {'verif ¢':>9s} "
         f"{'residuo dBFS':>13s} {'clic':>7s} {'MB':>7s} {'min':>5s}"]
    con_aviso = 0
    for r in filas:
        if r["estado"].startswith("ERROR"):
            L.append(f"{r['tema'][:28]:28s} {r['estado']}")
            continue
        hz = f"{r['hz']:.2f}" if r.get("hz") else "—"
        unc = f"{r['unc_cents']:.2f}" if r.get("unc_cents") is not None else "—"
        vc = r.get("verif_cents")
        verif = "—" if vc is None else f"{vc:+.2f}" + (" ⚠" if abs(vc) > UMBRAL_CENTS else "")
        aviso = " ⚠" if (r["residuo_dbfs"] >= RESIDUO_MAX_DBFS and not ES_MP3) else ""
        marca = " ⚠" if r.get("aviso_afinacion") and not verif.endswith("⚠") else ""
        con_aviso += bool(r.get("aviso_afinacion"))
        L.append(f"{r['tema'][:28]:28s} {r['estado']:10s} {hz:>7s} {unc:>5s} {r['correccion_cents']:>+8.2f} "
                 f"{FUENTE_CORTA.get(r.get('fuente', '—'), '—'):>7s} {verif:>9s}{marca} "
                 f"{r['residuo_dbfs']:>13.1f}{aviso} {_clic_corto(r.get('clic')):>7s} {r['mb']:>7.1f} "
                 f"{r.get('minutos', 0):>5.1f}")
    L.append("clic: BPM mediano del clic (o «manual» si viene de <tema>.clic.csv); detalle en <tema>/clic/clic_informe.txt.")
    L.append("fuente: med1/med2 = medición 1 o 2, comb = combinación de ambas, manual = <tema>.afinacion.txt; "
             "verif = afinación medida tras separar (solo modo completo).")
    if con_aviso:
        L.append(f"⚠ {con_aviso} tema(s) con aviso de afinación: revisa su afinacion.txt y escúchalos antes de usarlos.")
    L.append("Tramos de tempo: llegan con los clics (fase 2).")
    return "\n".join(L) + "\n"


# ============================ «solo clic» ============================
def reconstruir_mezcla(carpeta):
    """Suma los stems «NN Nombre» (NN ≥ 01) de un tema ya procesado. Por la resta exacta, esa suma es la
    mezcla afinada (con la ganancia aplicada). Se verifica contra «00 Mezcla afinada» y el manifiesto."""
    import re as _re
    man = json.loads((carpeta / ".proceso.json").read_text(encoding="utf-8"))
    stems = sorted(f for f in carpeta.iterdir() if f.is_file() and f.suffix in (".flac", ".mp3")
                   and _re.match(r"^(0[1-9]|[1-9]\d) ", f.name))
    if not stems:
        raise RuntimeError(f"{carpeta.name}: no hay stems para reconstruir la mezcla")
    sr = sf.info(str(stems[0])).samplerate if stems[0].suffix == ".flac" else int(man.get("frecuencia", 44100))
    suma = None
    for f in stems:
        y = leer_escrito(f)
        suma = y if suma is None else suma[:, :min(suma.shape[1], y.shape[1])] + y[:, :min(suma.shape[1], y.shape[1])]
    esperados = 3 if man.get("modo") == "karaoke" else (9 if man.get("acustica_electrica") else 8)
    ver = {"stems": len(stems), "esperados": esperados, "residuo_manifiesto": man["resumen"].get("residuo_dbfs"),
           "mp3": stems[0].suffix == ".mp3", "sr": sr, "n": int(suma.shape[1])}
    mezcla00 = next((f for f in carpeta.iterdir() if f.is_file() and f.name.startswith("00 ")), None)
    if mezcla00 is not None:
        ref = leer_escrito(mezcla00)
        n = min(ref.shape[1], suma.shape[1])
        ver["residuo_ahora"] = round(dbfs(pico(suma[:, :n] - ref[:, :n])), 1)
        ver["ok"] = ver["mp3"] or ver["residuo_ahora"] < RESIDUO_MAX_DBFS
    else:
        ver["ok"] = True
        ver["nota"] = "no se afinó: la suma de stems es la mezcla original"
    ver["ok"] = ver["ok"] and len(stems) == esperados
    return suma.astype(np.float32), sr, ver, man


def _texto_verificacion(v):
    t = f"suma de {v['stems']} stems (esperados {v['esperados']})"
    if "residuo_ahora" in v:
        t += f" vs «00 Mezcla afinada»: {v['residuo_ahora']} dBFS (manifiesto: {v['residuo_manifiesto']} dBFS)"
    else:
        t += f" · {v.get('nota', '')}"
    return t + (" · OK" if v["ok"] else " · ⚠ NO CUADRA: revisa los stems de este tema")


def descomprimir_tema_seguro(zip_ruta, tema, destino_base):
    """Kaggle: descomprime <tema>.zip (de /kaggle/input, solo lectura) en /kaggle/temp. Solo acepta rutas
    «<tema>/…» sin «..» ni rutas absolutas."""
    import zipfile
    carpeta = destino_base / tema
    if carpeta.exists():
        borrar_seguro(carpeta, tema_en_curso=tema)
    with zipfile.ZipFile(zip_ruta) as z:
        for info in z.infolist():
            nombre = info.filename
            partes = Path(nombre).parts
            if info.is_dir() or not nombre.startswith(f"{tema}/") or ".." in partes or nombre.startswith("/"):
                continue
            destino = destino_base.joinpath(*partes)
            destino.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(destino, "wb") as dst:
                shutil.copyfileobj(src, dst)
    return carpeta


def clic_tema_procesado(item):
    """«solo clic»: item = carpeta del tema (Colab) o <tema>.zip (Kaggle). No separa ni afina."""
    t0 = time.time()
    tema = item.stem if item.is_file() else item.name
    tiempos = {}
    trabajo = TRABAJO / f"{tema}__clic"
    if trabajo.exists():
        borrar_seguro(trabajo)
    trabajo.mkdir(parents=True)
    extraida = None
    try:
        if item.is_file():  # Kaggle: el zip viene de un Input (solo lectura)
            with Etapa("descomprimir en /kaggle/temp", tiempos):
                extraida = descomprimir_tema_seguro(item, tema, SALIDA)
            carpeta = extraida
        else:
            carpeta = item
        with Etapa("reconstruir mezcla (suma de stems)", tiempos):
            mezcla, sr, ver, man = reconstruir_mezcla(carpeta)
            bat = next((f for f in carpeta.iterdir() if f.is_file() and f.name.startswith("03 Bateria")), None)
            bateria = leer_escrito(bat).astype(np.float32) if bat is not None else None
        print(f"    {_texto_verificacion(ver)}")
        with Etapa("clic (Beat This!)", tiempos):
            clic = generar_clic(tema, mezcla, sr, ver["n"], trabajo, ENTRADA if ZIPS is None else None,
                                bateria, sr_salida=sr, mp3=ver["mp3"])
            clic["verificacion_suma"] = ver
        print(f"    clic: {clic['pulsos']} pulsos · {clic['bpm_mediana']} BPM · {clic['zonas_dudosas']} zona(s) dudosa(s) "
              f"· desfase {clic['desfase_muestras']} muestras · {clic['fuente']}")
        with Etapa("guardar clic", tiempos):
            if ZIPS is not None:  # Kaggle: <tema>.clic.zip solo con la carpeta clic/ (sin re-empaquetar stems)
                zip_clic = ZIPS / f"{tema}.clic.zip"
                if zip_clic.exists():
                    mover_zip_a_anteriores(zip_clic, f"{tema}.clic")
                comprimir_tema(trabajo / "clic", ZIPS, nombre_zip=zip_clic.name, prefijo=f"{tema}/clic")
                if not verificar_zip(zip_clic, trabajo / "clic", prefijo=f"{tema}/clic"):
                    raise RuntimeError(f"El zip del clic de {tema} no se verificó")
            else:  # Colab: <tema>/clic/ en Drive; un clic anterior se MUEVE a _clic_anteriores/
                destino = carpeta / "clic"
                if destino.exists():
                    anteriores = carpeta / "_clic_anteriores"
                    anteriores.mkdir(exist_ok=True)
                    os.rename(destino, anteriores / f"clic_{time.strftime('%Y-%m-%d_%H%M%S')}")
                shutil.copytree(trabajo / "clic", destino)
                man["clic"] = clic
                (carpeta / ".proceso.json").write_text(json.dumps(man, ensure_ascii=False, indent=2), encoding="utf-8")
        return {"tema": tema, "estado": "listo", "clic": _resumen_clic(clic), "verificacion": ver,
                "minutos": round((time.time() - t0) / 60, 1)}
    finally:
        borrar_seguro(trabajo)
        if extraida is not None and extraida.exists():
            borrar_seguro(extraida, tema_en_curso=tema)  # copia descomprimida en /kaggle/temp


def texto_resumen_clic(filas):
    L = [f"{'tema':30s} {'estado':8s} {'fuente':8s} {'pulsos':>6s} {'BPM':>6s} {'dudosas':>7s} {'desfase':>7s}  suma de stems"]
    for r in filas:
        if r["estado"].startswith("ERROR"):
            L.append(f"{r['tema'][:30]:30s} {r['estado']}")
            continue
        c, v = r["clic"], r["verificacion"]
        L.append(f"{r['tema'][:30]:30s} {r['estado']:8s} {'manual' if c['manual'] else 'auto':8s} {c['pulsos']:>6d} "
                 f"{c['bpm'] if c['bpm'] else '—':>6} {c['zonas']:>7d} {str(c['desfase']):>7s}  "
                 + ("OK" if v["ok"] else "⚠ no cuadra"))
    L.append("fuente: auto = Beat This!, manual = <tema>.clic.csv · desfase = muestras entre el clic y los stems "
             "(0 = alineado) · zonas dudosas: ver clic/clic_informe.txt de cada tema.")
    return "\n".join(L) + "\n"


def etapas_solo_clic():
    return 3 + (1 if ZIPS is not None else 0)


if ENTORNO == "kaggle" and ACCION != "solo clic":  # revisión de espacio con los modelos ya descargados
    _mb = [estimar_mb_tema(duracion_segundos(p)) for p in TEMAS if not tema_terminado(p.stem) or en_reprocesar(p.stem)]
    revisar_espacio_kaggle(WORKING, TEMP, sum(_mb), max(_mb, default=0), 0)

RESUMEN = []
PROGRESO = Progreso(len(TEMAS), SALIDA_INFO / "progreso.txt", mostrar=MOSTRAR_AVANCE, lineas_texto=ENTORNO == "kaggle")
_SOLO_CLIC = ACCION == "solo clic"
for _i, _ruta in enumerate(TEMAS, start=1):
    _tema = _ruta.stem if _ruta.is_file() else _ruta.name
    print(f"\n━━━ {_i}/{len(TEMAS)} · {_ruta.name} ━━━")
    PROGRESO.nuevo_tema(_i, _tema, etapas_solo_clic() if _SOLO_CLIC else etapas_previstas())
    try:
        RESUMEN.append(clic_tema_procesado(_ruta) if _SOLO_CLIC else procesar_tema(_ruta))
    except Detenido:
        raise
    except Exception as e:
        traceback.print_exc()
        RESUMEN.append({"tema": _tema, "estado": f"ERROR: {type(e).__name__}: {e}"})
        with open(SALIDA_INFO / "errores.txt", "a", encoding="utf-8") as _f:
            _f.write(f"{time.strftime('%Y-%m-%d %H:%M')} · {_ruta.name}\n{traceback.format_exc()}\n")
        print("  ✗ falló este tema (detalle en salida/errores.txt); sigo con el siguiente")
    PROGRESO.fin_tema()
    gc.collect()

_texto = texto_resumen_clic(RESUMEN) if _SOLO_CLIC else texto_resumen(RESUMEN)
if ZIPS is not None:
    _zips = sorted(ZIPS.glob("*.zip"))
    _texto += ("\nZips para descargar (pestaña Output › stems/zips):\n"
               + "".join(f"  {z.name}  ({z.stat().st_size / 1e6:.0f} MB)\n" for z in _zips))
PROGRESO.cerrar(_texto)
PROGRESO = None
print("\n" + "═" * 96)
print(_texto)


## 10 · (Opcional) Descargar un tema

In [ ]:
#@title 10 · (Opcional) Descargar un tema como .zip
TEMA_A_DESCARGAR = ""  #@param {type:"string"}
#@markdown Escribe el nombre de la carpeta del tema (tal como aparece en salida/) y ejecuta solo esta celda.
if TEMA_A_DESCARGAR.strip():
    _carpeta = SALIDA / TEMA_A_DESCARGAR.strip()
    if not _carpeta.is_dir():
        print(f"No existe {_carpeta}. Temas disponibles:")
        for _c in sorted(p.name for p in SALIDA.iterdir() if p.is_dir()):
            print("   ·", _c)
    elif globals().get("ENTORNO") == "kaggle":
        print("En Kaggle los zips ya quedan en /kaggle/working/stems/zips/: descárgalos desde la pestaña Output.")
    else:
        _zip = shutil.make_archive(str(TRABAJO / _carpeta.name), "zip", root_dir=_carpeta)
        from google.colab import files
        files.download(_zip)
        print("✓ Descarga iniciada:", Path(_zip).name)
